In [1]:
import requests, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 200)

def getFMPData(endpoint, urlhead=None, apikey=None, **params):
    if urlhead is None:
        urlhead = "https://financialmodelingprep.com/stable/"
    if apikey is None:
        apikey = os.getenv("FMP_API_KEY")

    if "from_" in params:
        params["from"] = params.pop("from_")

    url = urlhead + endpoint
    param_lst = [f"apikey={apikey}"] + [str(k) + "=" + str(v) for k, v in params.items()]
    url = "?".join([url, "&".join(param_lst)])
    data = requests.get(url).json()
    return data


from dotenv import load_dotenv
load_dotenv()

In [2]:
!pwd

/home/sagemaker-user/QuantTrading/BuildQuantFeatures


In [3]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("../").resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [4]:
import util
print(util.__file__)

/home/sagemaker-user/QuantTrading/util/__init__.py


# Load Dataset

In [5]:
df = pd.read_parquet("s3://sagemaker-us-east-1-209479286572/datasets/QuantTradingModelData/data_price-profile-ev-incm-bal-cf-km_sampled100.parquet")
df.sort_values(['symbol', 'date'], inplace=True)
df.shape

(275933, 196)

In [6]:
df.tail()

,symbol,date,adjOpen,adjHigh,adjLow,adjClose,adjVolume,rawClose,rawVolume,exchange,industry,sector,ipoDate,stockPrice,numberOfShares,marketCapitalization,minusCashAndCashEquivalents,addTotalDebt,evEnterpriseValue,price_tr20,dollarVolume,dollarVolume_tr20,turnOver,turnOver_tr20,lowLiquidity,incmFilingDate,incmAcceptedDate,revenue,costOfRevenue,grossProfit,researchAndDevelopmentExpenses,generalAndAdministrativeExpenses,sellingAndMarketingExpenses,sellingGeneralAndAdministrativeExpenses,otherExpenses,operatingExpenses,costAndExpenses,netInterestIncome,interestIncome,interestExpense,incmDepreciationAndAmortization,ebitda,ebit,nonOperatingIncomeExcludingInterest,operatingIncome,totalOtherIncomeExpensesNet,incomeBeforeTax,incomeTaxExpense,netIncomeFromContinuingOperations,netIncomeFromDiscontinuedOperations,otherAdjustmentsToNetIncome,incmNetIncome,netIncomeDeductions,bottomLineNetIncome,eps,epsDiluted,weightedAverageShsOut,weightedAverageShsOutDil,balFilingDate,balAcceptedDate,cashAndCashEquivalents,shortTermInvestments,cashAndShortTermInvestments,netReceivables,balAccountsReceivables,otherReceivables,balInventory,prepaids,otherCurrentAssets,totalCurrentAssets,propertyPlantEquipmentNet,goodwill,intangibleAssets,goodwillAndIntangibleAssets,longTermInvestments,taxAssets,otherNonCurrentAssets,totalNonCurrentAssets,otherAssets,totalAssets,totalPayables,accountPayables,otherPayables,accruedExpenses,shortTermDebt,capitalLeaseObligationsCurrent,taxPayables,deferredRevenue,otherCurrentLiabilities,totalCurrentLiabilities,longTermDebt,capitalLeaseObligationsNonCurrent,deferredRevenueNonCurrent,deferredTaxLiabilitiesNonCurrent,otherNonCurrentLiabilities,totalNonCurrentLiabilities,otherLiabilities,capitalLeaseObligations,totalLiabilities,treasuryStock,preferredStock,commonStock,retainedEarnings,additionalPaidInCapital,accumulatedOtherComprehensiveIncomeLoss,otherTotalStockholdersEquity,totalStockholdersEquity,totalEquity,minorityInterest,totalLiabilitiesAndTotalEquity,totalInvestments,totalDebt,netDebt,cfFilingDate,cfAcceptedDate,cfNetIncome,cfDepreciationAndAmortization,deferredIncomeTax,stockBasedCompensation,changeInWorkingCapital,cfAccountsReceivables,cfInventory,accountsPayables,otherWorkingCapital,otherNonCashItems,netCashProvidedByOperatingActivities,investmentsInPropertyPlantAndEquipment,acquisitionsNet,purchasesOfInvestments,salesMaturitiesOfInvestments,otherInvestingActivities,netCashProvidedByInvestingActivities,netDebtIssuance,longTermNetDebtIssuance,shortTermNetDebtIssuance,netStockIssuance,netCommonStockIssuance,commonStockIssuance,commonStockRepurchased,netPreferredStockIssuance,netDividendsPaid,commonDividendsPaid,preferredDividendsPaid,otherFinancingActivities,netCashProvidedByFinancingActivities,effectOfForexChangesOnCash,netChangeInCash,cashAtEndOfPeriod,cashAtBeginningOfPeriod,operatingCashFlow,capitalExpenditure,freeCashFlow,incomeTaxesPaid,interestPaid,marketCap,kmEnterpriseValue,evToSales,evToOperatingCashFlow,evToFreeCashFlow,evToEBITDA,netDebtToEBITDA,currentRatio,incomeQuality,grahamNetNet,taxBurden,interestBurden,workingCapital,investedCapital,returnOnAssets,operatingReturnOnAssets,returnOnTangibleAssets,returnOnEquity,returnOnInvestedCapital,returnOnCapitalEmployed,earningsYield,freeCashFlowYield,capexToOperatingCashFlow,capexToDepreciation,capexToRevenue,salesGeneralAndAdministrativeToRevenue,researchAndDevelopementToRevenue,stockBasedCompensationToRevenue,intangiblesToTotalAssets,averageReceivables,averagePayables,averageInventory,daysOfSalesOutstanding,daysOfPayablesOutstanding,daysOfInventoryOutstanding,operatingCycle,cashConversionCycle,freeCashFlowToEquity,freeCashFlowToFirm,tangibleAssetValue,netCurrentAssetValue,grahamNumber
275928,ZK,2025-12-17,26.74,26.77,26.66,26.66,354521,26.66,354521.0,NYSE,Auto - Manufacturers,Consumer Cyclical,2024-05-10,216.97,257296840.0,5.582618e+10,7.021000e+09,3.162400e+10,8.042918e+10,26.7870,9451529.86,9.628350e+06,0.001378,0.001396,0,2025-11-17,2025-11-17 06:58:31,3.156200e+10

In [4]:
df['date'].min().strftime("%Y-%m-%d")

'2000-01-03'

In [6]:
df['date']

0        2016-02-24
1        2016-02-25
2        2016-02-26
3        2016-02-29
4        2016-03-01
            ...    
275928   2025-12-17
275929   2025-12-18
275930   2025-12-19
275931   2025-12-22
275932   2025-12-23
Name: date, Length: 275933, dtype: datetime64[ns]

In [5]:
spy = getFMPData('historical-price-eod/dividend-adjusted', symbol='SPY', from_='2000-01-01', to='2026-01-31')
spy = pd.DataFrame(spy)
spy['date'] = pd.to_datetime(spy['date'])
spy.set_index('date', inplace=True)
spy_closePrice = spy['adjClose'].sort_index()

spy_closePrice

date
2006-03-17     90.28
2006-03-20     90.14
2006-03-21     89.57
2006-03-22     90.11
2006-03-23     89.93
               ...  
2026-01-26    690.84
2026-01-27    693.59
2026-01-28    693.52
2026-01-29    692.15
2026-01-30    690.08
Name: adjClose, Length: 5000, dtype: float64

In [6]:
df['symbol'].nunique()

100

# Feature Builder

In [6]:
from collections import defaultdict


epsilon = 1e-6
featureNames, featureValues = defaultdict(list), defaultdict(pd.Series)

df['ipoDate'] = np.where(df['ipoDate'].notna(), df['ipoDate'], df.groupby('symbol')['date'].transform("min"))
df['daysToIPO'] = (df['date'] - df['ipoDate']).dt.days.astype(float)
df['daysToFirstRec'] = (df['date'] - df.groupby('symbol')['date'].transform("min")).dt.days.astype(float)
df['daysToLastIncm'] = (df['date'] - df['incmFilingDate']).dt.days.astype(float)
df['daysToLastBal'] = (df['date'] - df['balFilingDate']).dt.days.astype(float)
df['daysToLastCf'] = (df['date'] - df['cfFilingDate']).dt.days.astype(float)
df['rn'] = df.groupby("symbol").cumcount() + 1
df['rn_max'] = df.groupby('symbol')['rn'].transform('max')

df['eligible'] = np.where(
    (df['lowLiquidity'] == 0) &
    (df['rn'] > 120) &
    (df['rn'] < df['rn_max']),
    1, 0
)

featureNames['filter'] = ['eligible', 'lowLiquidity', 'rn']
featureNames['date'] = ['daysToIPO', 'daysToLastIncm', 'daysToLastBal', 'daysToLastCf']


In [3]:
from numpy.lib.stride_tricks import sliding_window_view
from __future__ import annotations
from dataclasses import dataclass, field
from collections.abc import Sized
from collections.abc import Iterable
from collections import defaultdict
from itertools import product, combinations, permutations
from typing import Any, Callable, Iterable, Optional, Tuple
from functools import partial
from tqdm import tqdm
import math, re


def is_container(obj):
    return (
    isinstance(obj, Iterable)
    and isinstance(obj, Sized)
    and not isinstance(obj, (str, bytes))
    )


def unwrap_single(obj):
    if is_container(obj) and len(obj) == 1:
        if isinstance(obj, dict):
            return next(iter(obj.values()))
        return next(iter(obj))
    else:
        raise ValueError("input is not a container or has more than 1 elements")


def _as_sized(seq: Iterable[Any]) -> Any:
    """
    If seq has __len__, return as-is.
    Otherwise materialize to a tuple so we can compute n reliably.
    """
    try:
        len(seq)  # type: ignore[arg-type]
        return seq
    except TypeError:
        return tuple(seq)


class Combinator:
    """
    Wrapper over itertools.{product, combinations, permutations} that can:
      - preset r for combinations/permutations at construction time
      - override r at call time
      - compute output length without enumerating (materializes only if needed)
    """

    def __init__(self, combine: str = "product", *, r: Optional[int] = None, **kwargs):
        self.combine = combine.lower()
        self.r = r
        self._init_kwargs = dict(kwargs)

        if self.combine == "product":
            # product(*iterables, repeat=...)
            self.fn = partial(product, **self._init_kwargs)

        elif self.combine in ("combination", "combinations"):
            # we will supply r at call time (preset or override)
            self.combine = "combinations"
            self.fn = combinations  # don't partial r here; we want it flexible

        elif self.combine in ("permutation", "permutations"):
            self.combine = "permutations"
            self.fn = permutations  # same idea

        else:
            raise ValueError(f"Unknown combine type: {combine}")

    def __call__(self, *args, **kwargs):
        """
        product:        builder(*iterables, repeat=...)
        combinations:   builder(iterable, r=...)   OR builder(iterable) if preset r
        permutations:   builder(iterable, r=...)   OR builder(iterable) if preset r
        """
        if self.combine == "product":
            # allow overriding repeat at call time
            merged = dict(self._init_kwargs)
            merged.update(kwargs)
            return product(*args, **merged)

        if len(args) != 1:
            raise TypeError(f"{self.combine} expects exactly 1 positional arg: (iterable,)")

        iterable = args[0]
        r = kwargs.pop("r", None)
        if kwargs:
            raise TypeError(f"Unexpected keyword args for {self.combine}: {list(kwargs.keys())}")

        r = self.r if r is None else int(r)
        if r is None:
            raise TypeError(f"{self.combine} requires r (preset in constructor or pass r=...)")

        return self.fn(iterable, r)

    def getLen(self, *args, **kwargs) -> int:
        """
        Compute number of yielded items without enumerating.

        product:        getLen(*iterables, repeat=...)
        combinations:   getLen(iterable, r=...)    OR getLen(iterable) if preset r
        permutations:   getLen(iterable, r=...)    OR getLen(iterable) if preset r
        """
        if self.combine == "product":
            repeat = kwargs.get("repeat", self._init_kwargs.get("repeat", 1))
            if repeat is None:
                repeat = 1
            repeat = int(repeat)

            if len(args) == 0:
                return 1  # product() yields one empty tuple

            lens = []
            for it in args:
                it2 = _as_sized(it)
                lens.append(len(it2))  # type: ignore[arg-type]

            base = math.prod(lens) if lens else 1
            return base ** repeat

        if len(args) != 1:
            raise TypeError(f"getLen for {self.combine} expects exactly 1 positional arg: (iterable,)")

        iterable = args[0]
        seq = _as_sized(iterable)
        n = len(seq)  # type: ignore[arg-type]

        r = kwargs.pop("r", None)
        if kwargs:
            raise TypeError(f"Unexpected keyword args for getLen({self.combine}): {list(kwargs.keys())}")

        if self.combine == "combinations":
            r = self.r if r is None else int(r)
            if r is None:
                raise TypeError("getLen for combinations requires r (preset in constructor or pass r=...)")
            if r < 0 or r > n:
                return 0
            return math.comb(n, r)

        if self.combine == "permutations":
            if r is None and self.r is None:
                r = n  # match itertools.permutations default
            else:
                r = self.r if r is None else int(r)

            if r < 0 or r > n:
                return 0
            return math.perm(n, r)

        raise ValueError(f"Unknown combine type: {self.combine}")



class FeatureGenerator:
    def __init__(self, df, **kwargs):
        self.df = df
        self.rawInput = {
            'openPrice': kwargs.get('open') or 'adjOpen',
            'closePrice': kwargs.get('close') or 'adjClose',
            'highPrice': kwargs.get('high') or 'adjHigh',
            'lowPrice': kwargs.get('low') or 'adjLow',
            'volume': kwargs.get('volume') or 'adjVolume',
            'ticker': kwargs.get('ticker') or 'symbol',
        }
        self.cache = {
            'values': defaultdict(pd.Series),
            'names': defaultdict(list)
        }
        self.eps = kwargs.get('eps', 1e-10)
        self.max_window = kwargs.get('max_window', 127)
        self.schema = None
        self.df['date'] = pd.to_datetime(self.df['date'])
        self.minDate = self.df['date'].min()
        self.maxDate = self.df['date'].max()

    def cache_column(self, colVal, colName, famName):
        colVal.name = colName
        self.cache['values'][colName] = colVal
        self.cache['names'][famName].append(colName)

    def build_features(self, schema: dict, max_window = None):
        self.schema = schema
        self.max_window = max_window if max_window is not None else self.max_window
        for key, val in self.schema.items():
            famName = val['name']
            create_fn = getattr(self, f'create_{key}', None)
            if create_fn is None:
                raise ValueError(f"Unknown feature: {key}")
            param = val.get('parameters', {})

            parameters = []
            for k, v in param.items():
                parameter = self.parse_parameters(v)
                parameters.append(parameter)
            # print(parameters)

            combinator = self.schema[key].get('param_combine', Combinator('product'))
            iter_p = combinator(*parameters)
            iter_n = combinator.getLen(*parameters)
            for p in tqdm(iter_p, total=iter_n, desc=f'Calculating {famName}'):
                # print(p)
                create_fn(*p)

    def parse_parameters(self, p):
        if is_container(p):
            return p
        elif isinstance(p, str) and bool(re.fullmatch(r"[^-]+-[^-]+", p)):
            p0, p1 = p.split('-')
            return self.schema[p0]['parameters'][p1]
        elif isinstance(p, str) and (p in self.df.columns or p in self.cache['values']):
            return [p]
        elif isinstance(p, str) and p in self.cache['names']:
            return self.cache['names'][p]
        elif isinstance(p, pd.Series) or isinstance(p, np.ndarray):
            return [p]
        else:
            raise ValueError("Unknown input parameter types")

    def create_ret(self, w):
        colName = f'ret_{w}'
        if colName in self.cache['values']:
            return self.cache['values'][colName]

        if w <= self.max_window:
            x = self.df[self.rawInput['closePrice']]
            g = self.df[self.rawInput['ticker']]
            col = np.log1p(x.groupby(g).pct_change(w))
            self.cache_column(col, colName, self.schema['ret']['name'])
            return col

    def create_retNext(self, w):
        colName = f'retNext_{w}'
        if colName in self.cache['values']:
            return self.cache['values'][colName]

        x = self.create_ret(w)
        g = self.df[self.rawInput['ticker']]
        col = x.groupby(g).shift(-w)
        self.cache_column(col, colName, self.schema['retNext']['name'])
        return col

    def create_mar(self, i, j):
        colName = f'mar_{i}_{j}'
        if colName in self.cache['values']:
            return self.cache['values'][colName]

        if i + j <= self.max_window:
            x = self.create_ret(i)
            g = self.df[self.rawInput['ticker']]
            col = x.groupby(g).transform(lambda s: s.rolling(j).mean())
            self.cache_column(col, colName, self.schema['mar']['name'])
            return col

    def create_mvr(self, i, j):
        colName = f'mvr_{i}_{j}'
        if colName in self.cache['values']:
            return self.cache['values'][colName]

        if i + j <= self.max_window:
            x = self.create_ret(i)
            g = self.df[self.rawInput['ticker']]
            col = x.groupby(g).transform(lambda s: s.rolling(j).std())
            self.cache_column(col, colName, self.schema['mvr']['name'])
            return col

    def create_mrz(self, i, j):
        colName = f'mrz_{i}_{j}'
        if colName in self.cache['values']:
            return self.cache['values'][colName]

        if i + j <= self.max_window and i < j:
            x = self.create_ret(i)
            mu = self.create_mar(i, j)
            std = self.create_mvr(i, j)
            col = (x - mu) / (std + self.eps)
            self.cache_column(col, colName, self.schema['mrz']['name'])
            return col

    def create_mrr(self, i, j):
        colName = f'mrr_{i}_{j}'
        if colName in self.cache['values']:
            return self.cache['values'][colName]

        if i + j <= self.max_window and i < j:
            x = self.create_ret(i)
            mu = self.create_mar(i, j)
            col = x / (mu + self.eps)
            self.cache_column(col, colName, self.schema['mrr']['name'])
            return col

    def create_map(self, w):
        colName = f'map_{w}'
        if colName in self.cache['values']:
            return self.cache['values'][colName]

        if w <= self.max_window:
            x = self.df[self.rawInput['closePrice']]
            g = self.df[self.rawInput['ticker']]
            col = x.groupby(g).transform(lambda s: s.rolling(w).mean())
            self.cache_column(col, colName, self.schema['map']['name'])
            return col

    def create_mvp(self, w):
        colName = f'mvp_{w}'
        if colName in self.cache['values']:
            return self.cache['values'][colName]

        if w <= self.max_window:
            x = self.df[self.rawInput['closePrice']]
            g = self.df[self.rawInput['ticker']]
            col = x.groupby(g).transform(lambda s: s.rolling(w).std())
            self.cache_column(col, colName, self.schema['mvp']['name'])
            return col

    def create_blgz(self, w):
        colName = f'blgz_{w}'
        if colName in self.cache['values']:
            return self.cache['values'][colName]

        if w <= self.max_window:
            p = self.df[self.rawInput['closePrice']]
            mu = self.create_map(w)
            std = self.create_mvp(w)
            col = (p - mu) / (std + self.eps)
            col = col.clip(-5, 5)
            self.cache_column(col, colName, self.schema['blgz']['name'])
            return col

    def create_tr(self):
        colName = f'tr'
        if colName in self.cache['values']:
            return self.cache['values'][colName]

        col = np.maximum(
            self.df[self.rawInput['highPrice']] - self.df[self.rawInput['lowPrice']],
            np.abs(self.df[self.rawInput['highPrice']] - self.df[self.rawInput['closePrice']].shift(1)),
            np.abs(self.df[self.rawInput['lowPrice']] - self.df[self.rawInput['closePrice']].shift(1))
        )
        self.cache_column(col, colName, self.schema['tr']['name'])
        return col

    def create_atr(self, w):
        colName = f'atr_{w}'
        if colName in self.cache['values']:
            return self.cache['values'][colName]

        if w <= self.max_window:
            x = self.create_tr()
            g = self.df[self.rawInput['ticker']]
            col = x.groupby(g).transform(lambda s: s.rolling(w).mean())
            self.cache_column(col, colName, self.schema['atr']['name'])
            return col

    def create_vtr(self, w):
        colName = f'vtr_{w}'
        if colName in self.cache['values']:
            return self.cache['values'][colName]

        if w <= self.max_window:
            x = self.create_tr()
            g = self.df[self.rawInput['ticker']]
            col = x.groupby(g).transform(lambda s: s.rolling(w).std())
            self.cache_column(col, colName, self.schema['vtr']['name'])
            return col

    def create_natr(self, w):
        colName = f'natr_{w}'
        if colName in self.cache['values']:
            return self.cache['values'][colName]

        if w <= self.max_window:
            atr = self.create_atr(w)
            mapr = self.create_map(w)
            col = atr / (mapr + self.eps)
            self.cache_column(col, colName, self.schema['natr']['name'])
            return col

    def create_zatr(self, i, j):
        colName = f'zatr_{i}_{j}'
        if colName in self.cache['values']:
            return self.cache['values'][colName]

        if i + j <= self.max_window and i < j:
            atr = self.create_atr(i)
            g = self.df[self.rawInput['ticker']]
            mu = atr.groupby(g).transform(lambda s: s.rolling(j).mean())
            std = atr.groupby(g).transform(lambda s: s.rolling(j).std())
            col = (atr - mu) / (std + self.eps)
            self.cache_column(col, colName, self.schema['zatr']['name'])
            return col

    def create_ratr(self, i, j):
        colName = f'ratr_{i}_{j}'
        if colName in self.cache['values']:
            return self.cache['values'][colName]

        if i + j <= self.max_window and i < j:
            atr = self.create_atr(i)
            g = self.df[self.rawInput['ticker']]
            mu = atr.groupby(g).transform(lambda s: s.rolling(j).mean())
            std = atr.groupby(g).transform(lambda s: s.rolling(j).std())
            col = atr / (mu + self.eps)
            self.cache_column(col, colName, self.schema['ratr']['name'])
            return col

    def create_mapd(self, i, j):
        colName = f'mapd_{i}_{j}'
        if colName in self.cache['values']:
            return self.cache['values'][colName]

        short = self.create_map(i)
        long = self.create_map(j)
        col = short - long
        self.cache_column(col, colName, self.schema['mapd']['name'])
        return col

    def create_mapdln(self, i, j):
        colName = f'mapdln_{i}_{j}'
        if colName in self.cache['values']:
            return self.cache['values'][colName]

        short = self.create_map(i)
        long = self.create_map(j)
        col = np.log(short + self.eps) - np.log(long + self.eps)
        self.cache_column(col, colName, self.schema['mapdln']['name'])
        return col

    def create_mapdrtp(self, i, j):
        colName = f'mapdrtp_{i}_{j}'
        if colName in self.cache['values']:
            return self.cache['values'][colName]

        diff = self.create_mapd(i, j)
        long = self.create_map(j)
        col = diff / (long + self.eps)
        self.cache_column(col, colName, self.schema['mapdrtp']['name'])
        return col

    def create_mapdrtatr(self, i, j):
        colName = f'mapdrtatr_{i}_{j}'
        if colName in self.cache['values']:
            return self.cache['values'][colName]

        diff = self.create_mapd(i, j)
        atr = self.create_atr(j)
        col = diff / (atr + self.eps)
        self.cache_column(col, colName, self.schema['mapdrtatr']['name'])
        return col

    def create_mapdlnrtmvr(self, i, j):
        colName = f'mapdlnrtmvr_{i}_{j}'
        if colName in self.cache['values']:
            return self.cache['values'][colName]

        diff = self.create_mapdln(i, j)
        mvr = self.create_mvr(1, j)
        col = diff / (mvr + self.eps)
        self.cache_column(col, colName, self.schema['mapdlnrtmvr']['name'])
        return col

    def create_pdiff(self, w):
        colName = f'pdiff_{w}'
        if colName in self.cache['values']:
            return self.cache['values'][colName]

        g = self.rawInput['ticker']
        col = self.df.groupby(g)[self.rawInput['closePrice']].diff(w)
        self.cache_column(col, colName, 'PriceDifference')
        return col

    def create_rsi(self, w):
        colName = f'rsi_{w}'
        if colName in self.cache['values']:
            return self.cache['values'][colName]

        g = self.df[self.rawInput['ticker']]
        pdiff = self.create_pdiff(1)
        gain = pdiff.clip(lower=0)
        loss = (-pdiff).clip(lower=0)

        avg_gain = gain.groupby(g).transform(lambda s: s.ewm(alpha=1/w, adjust=False, min_periods=w).mean())
        avg_loss = loss.groupby(g).transform(lambda s: s.ewm(alpha=1/w, adjust=False, min_periods=w).mean())
        rs = avg_gain / (avg_loss + self.eps)
        col = 100 - (100 / (1 + rs))

        self.cache_column(col, colName, self.schema['rsi']['name'])
        return col

    def create_rsiz(self, i, j):
        colName = f'rsiz_{i}_{j}'
        if colName in self.cache['values']:
            return self.cache['values'][colName]

        if i + j <= self.max_window and i < j:
            g = self.df[self.rawInput['ticker']]
            rsi = self.create_rsi(i)
            mu = rsi.groupby(g).transform(lambda s: s.rolling(j).mean())
            std = rsi.groupby(g).transform(lambda s: s.rolling(j).std())
            col = (rsi - mu) / (std + self.eps)
            self.cache_column(col, colName, self.schema['rsiz']['name'])
            return col

    def create_mdd(self, w):
        colName = f'mdd_{w}'
        if colName in self.cache['values']:
            return self.cache['values'][colName]

        g = self.df[self.rawInput['ticker']]
        rolling_peak = self.df.groupby(g)[self.rawInput['closePrice']].transform(lambda x: x.rolling(w).max())
        drawdown = self.df[self.rawInput['closePrice']] / rolling_peak - 1
        col = -(drawdown.groupby(g).transform(lambda x: x.rolling(w).min()))
        self.cache_column(col, colName, self.schema['mdd']['name'])
        return col

    def create_mddatrnorm(self, w):
        colName = f'mddatrnorm_{w}'
        if colName in self.cache['values']:
            return self.cache['values'][colName]

        g = self.df[self.rawInput['ticker']]
        mdd = self.create_mdd(w)
        atr = self.create_atr(14)
        atr_base = atr.groupby(g).transform(lambda x: x.ewm(span=252, adjust=False).mean())
        col = mdd / (atr_base + self.eps)
        self.cache_column(col, colName, self.schema['mddatrnorm']['name'])
        return col

    def create_baseprice(self, symbol):
        colName = f'baseprice_{symbol.lower()}'
        if colName in self.cache['values']:
            return self.cache['values'][colName]

        min_date = self.minDate.strftime("%Y-%m-%d")
        max_date = self.maxDate.strftime("%Y-%m-%d")
        endpoint = 'historical-price-eod/dividend-adjusted'
        data = getFMPData(
            endpoint,
            symbol=symbol.upper(),
            from_=min_date,
            to=max_date
        )
        df = pd.DataFrame(data)
        df['date'] = pd.to_datetime(df['date'])
        df.set_index('date', inplace=True)
        col = df['adjClose'].sort_index()
        self.cache_column(col, colName, 'MarketBaseline')
        return col

    def create_beta(self, w, symbol):
        colName = f'beta_{w}_{symbol.lower()}'
        if colName in self.cache['values']:
            return self.cache['values'][colName]

        if w <= self.max_window:
            mkt_p = self.create_baseprice(symbol)
            asset_df = self.df[['symbol', 'date', 'adjClose']]
            asset_p = pd.pivot(
                asset_df,
                index='date',
                columns='symbol',
                values='adjClose'
            ).sort_index()
            beta_keys = asset_df[['symbol', 'date']].copy()

            mkt_r = np.log1p(mkt_p.pct_change(fill_method=None))
            asset_r = np.log1p(asset_p.pct_change(fill_method=None))
            asset_r_aligned, mkt_r_aligned = asset_r.align(mkt_r, join='inner', axis=0)

            cov = asset_r_aligned.rolling(w, min_periods=w).cov(mkt_r_aligned)
            var = mkt_r_aligned.rolling(w, min_periods=w).var()
            beta_wide = cov.div(var.where(var.abs() > self.eps), axis=0)

            beta_long = beta_wide.stack(future_stack=True).rename(colName).reset_index()
            # beta_long.columns = ['date', 'symbol', colName]
            col = beta_keys.merge(beta_long, on=['symbol', 'date'], how='left', sort=False)[colName]
            self.cache_column(col, colName, self.schema['beta']['name'])
            return col

    def create_var(self, i, j, p):
        colName = f'var_{i}_{j}_p{int(p*100)}'
        if colName in self.cache['values']:
            return self.cache['values'][colName]

        if i + j <= self.max_window and i < j:
            g = self.df[self.rawInput['ticker']]
            ret = self.create_ret(i)
            # col = ret.groupby(g).rolling(j, min_periods=j).quantile(p)
            col = ret.groupby(g).transform(lambda s: s.rolling(j, min_periods=j).quantile(p))
            self.cache_column(col, colName, self.schema['var']['name'])
            return col

    def create_varr(self, i, j, p=0.05):
        colName = f'varr_{i}_{j}_p{int(p*100)}'
        if colName in self.cache['values']:
            return self.cache['values'][colName]

        if i + j <= self.max_window and i < j:
            lower = self.create_var(i, j, p)
            upper = self.create_var(i, j, 1-p)
            col = upper.abs() / (lower.abs() + self.eps)
            self.cache_column(col, colName, self.schema['varr']['name'])
            return col

    def create_cvar(self, i, j, p):
        colName = f'cvar_{i}_{j}_p{int(p*100)}'
        if colName in self.cache['values']:
            return self.cache['values'][colName]

        if i + j <= self.max_window and i < j and j * p >= 2:
            ret = self.create_ret(i)
            ret_long_df = pd.concat([self.df[[self.rawInput['ticker'], 'date']], ret], axis=1)
            ret_wide_df = pd.pivot(ret_long_df, columns=self.rawInput['ticker'], index='date', values=ret.name)
            q_arr = ret_wide_df.rolling(window=j, min_periods=j).quantile(p).to_numpy()
            q_arr = q_arr[(j - 1):, :]
            ret_arr = ret_wide_df.to_numpy()
            ret_arr_roll = sliding_window_view(ret_arr, j, axis=0)
            mask = ret_arr_roll < q_arr[:, :, None]
            sum_ = np.sum(np.where(mask, ret_arr_roll, 0.0), axis=2)
            cnt = np.sum(mask, axis=2)
            out = np.divide(sum_, cnt, out=np.full_like(sum_, np.nan), where=cnt > 0)
            out_wide_df = pd.DataFrame(out, index=ret_wide_df.iloc[(j - 1):].index, columns=ret_wide_df.columns)
            out_wide_df = out_wide_df.reindex(ret_wide_df.index)
            out_long_df = out_wide_df.reset_index().melt(
                id_vars=out_wide_df.index.name,
                var_name=out_wide_df.columns.name,
                value_name=colName
            )
            ret_long_df2 = ret_long_df.merge(out_long_df, how='left', on=[self.rawInput['ticker'], 'date'])
            col = ret_long_df2[colName]
            self.cache_column(col, colName, self.schema['cvar']['name'])
            return col


In [50]:
features = FeatureGenerator(df)
features.cache

{'values': defaultdict(pandas.core.series.Series, {}),
 'names': defaultdict(list, {})}

In [51]:
featureSchema = {
    'ret': {'name': 'LogReturn', 'parameters': {'w': [1, 3, 5, 10, 15, 20, 30, 50, 60, 90, 100, 120]}},
    'retNext': {'name': 'NextReturn', 'parameters': {'w': [1, 5, 10, 20, 50, 100]}},
    'mar': {'name': 'MovingAverageReturn', 'parameters': {'i': 'ret-w', 'j': [3, 5, 10, 15, 20, 30, 50, 100]}},
    'mvr': {'name': 'MovingVolatilityReturn', 'parameters': {'i': 'ret-w', 'j': 'mar-j'}},
    'mrz': {'name': 'MovingRetrunZscore', 'parameters': {'i': 'ret-w', 'j': [10, 20, 50, 100]}},
    'mrr': {'name': 'MovingRetrunRatio', 'parameters': {'i': 'ret-w', 'j': 'mrz-j'}},
    'map': {'name': 'MovingAveragePrice', 'parameters': {'w': [3, 5, 10, 15, 20, 30, 50, 100]}, 'hasUnit': True},
    'mvp': {'name': 'MovingVolatilityPrice', 'parameters': {'w': 'map-w'}, 'hasUnit': True},
    'blgz': {'name': 'BollingerZscore', 'parameters': {'w': 'map-w'}},
    'tr': {'name': 'TrueRange', 'parameters': {}, 'hasUnit': True},
    'atr': {'name': 'AverageTrueRange', 'parameters': {'w': 'map-w'}, 'hasUnit': True},
    'vtr': {'name': 'VolatilityTrueRange', 'parameters': {'w': 'map-w'}, 'hasUnit': True},
    'natr': {'name': 'NormalizedAverageTrueRange', 'parameters': {'w': 'map-w'}},
    'zatr': {'name': 'AverageTrueRangeZscore', 'parameters': {'i': 'map-w', 'j': [20, 50, 100]}},
    'ratr': {'name': 'AverageTrueRangeRatio', 'parameters': {'i': 'map-w', 'j': [20, 50, 100]}},
    'mapd': {'name': 'MovingAveragePriceDifference', 'parameters': {'w': 'map-w'}, 'param_combine': Combinator('combination', r=2), 'hasUnit': True},
    'mapdln': {'name': 'LogMovingAveragePriceDifference', 'parameters': {'w': 'map-w'}, 'param_combine': Combinator('combination', r=2), 'hasUnit': True},
    'mapdrtp': {'name': 'MovingAveragePriceDifferenceToPriceRatio', 'parameters': {'w': 'map-w'}, 'param_combine': Combinator('combination', r=2)},
    'mapdrtatr': {'name': 'MovingAveragePriceDifferenceToATRRatio', 'parameters': {'w': 'map-w'}, 'param_combine': Combinator('combination', r=2)},
    'mapdlnrtmvr': {'name': 'MovingAveragePriceDifferenceToVolatilityRatio', 'parameters': {'w': 'map-w'}, 'param_combine': Combinator('combination', r=2)},
    'rsi': {'name': 'RelativeStrengthIndex', 'parameters': {'w': [5, 10, 14, 15, 20, 30, 50]}, 'hasUnit': True},
    'rsiz': {'name': 'RelativeStrengthIndexZscore', 'parameters': {'i': 'rsi-w', 'j': [20, 50, 100]}},
    'mdd': {'name': 'MaxDrawDown', 'parameters': {'w': [10, 14, 20, 50, 60, 100]}},
    'mddatrnorm': {'name': 'VolatilityNormalizedMaxDrawDown', 'parameters': {'w': 'mdd-w'}},
    'beta': {'name': 'Beta', 'parameters': {'w': [5, 10, 14, 15, 20, 30, 50, 60, 126], 'baseline': ['SPY']}},
    # 'beta': {'name': 'Beta', 'parameters': {'w': [14, 20, 60, 126], 'baseline': ['SPY']}},
    'var': {'name': 'ValueAtRisk', 'parameters': {'i': [1], 'j': [10, 20, 30, 60, 126], 'p': [0.05, 0.95]}},
    'varr': {'name': 'ValueAtRiskRatio', 'parameters': {'i': [1], 'j': [10, 20, 30, 60, 126], 'p': [0.1, 0.05, 0.01]}},
    'cvar': {'name': 'ExpectedShortfall', 'parameters': {'i': [1], 'j': [30, 60, 126], 'p': [0.2, 0.1, 0.05]}},
}

features.build_features(featureSchema)

Calculating LogMovingAveragePriceDifference: 100%|██████████| 28/28 [00:00<00:00, 234.93it/s]
Calculating MovingAveragePriceDifferenceToPriceRatio: 100%|██████████| 28/28 [00:00<00:00, 490.70it/s]
Calculating MovingAveragePriceDifferenceToATRRatio: 100%|██████████| 28/28 [00:00<00:00, 507.97it/s]
Calculating MovingAveragePriceDifferenceToVolatilityRatio: 100%|██████████| 28/28 [00:00<00:00, 511.14it/s]
Calculating ExpectedShortfall: 100%|██████████| 9/9 [00:08<00:00,  1.12it/s]


In [52]:
for k, v in features.cache['names'].items():
    print(k, len(v))

LogReturn 12
NextReturn 6
MovingAverageReturn 82
MovingVolatilityReturn 82
MovingRetrunZscore 21
MovingRetrunRatio 21
MovingAveragePrice 8
MovingVolatilityPrice 8
BollingerZscore 8
TrueRange 1
AverageTrueRange 9
VolatilityTrueRange 8
NormalizedAverageTrueRange 8
AverageTrueRangeZscore 15
AverageTrueRangeRatio 15
MovingAveragePriceDifference 28
LogMovingAveragePriceDifference 28
MovingAveragePriceDifferenceToPriceRatio 28
MovingAveragePriceDifferenceToATRRatio 28
MovingAveragePriceDifferenceToVolatilityRatio 28
PriceDifference 1
RelativeStrengthIndex 7
RelativeStrengthIndexZscore 15
MaxDrawDown 6
VolatilityNormalizedMaxDrawDown 6
MarketBaseline 1
Beta 9
ValueAtRisk 30
ValueAtRiskRatio 15
ExpectedShortfall 8


In [39]:
features.cache['names']['ExpectedShortfall']

['cvar_1_30_p20',
 'cvar_1_30_p10',
 'cvar_1_60_p20',
 'cvar_1_60_p10',
 'cvar_1_60_p5',
 'cvar_1_126_p20',
 'cvar_1_126_p10',
 'cvar_1_126_p5']

In [40]:
features.cache['values']['cvar_1_60_p20'].isna().mean()

0.03390315765058909

In [43]:
ret_1 = features.cache['values']['ret_1']
cvar_1_60_p5 = features.cache['values']['cvar_1_60_p5']

In [49]:
r = ret_1.iloc[-60:]
r[r<r.quantile(0.05)].mean()

-0.03672045108435763

In [45]:
cvar_1_60_p5

0             NaN
1             NaN
2             NaN
3             NaN
4             NaN
           ...   
275928   -0.03672
275929   -0.03672
275930   -0.03672
275931   -0.03672
275932   -0.03672
Name: cvar_1_60_p5, Length: 275933, dtype: float64

In [41]:
features.cache['values']['ret_1']

0              NaN
1         0.000000
2         0.158224
3         0.047628
4         0.000000
            ...   
275928   -0.003744
275929   -0.002253
275930    0.004875
275931    0.000000
275932    0.000000
Name: ret_1, Length: 275933, dtype: float64

In [31]:
var_1_20_p20 = features.cache['values']['var_1_20_p20']
var_1_20_p20

0              NaN
1              NaN
2              NaN
3              NaN
4              NaN
            ...   
275928   -0.003135
275929   -0.003135
275930   -0.002690
275931   -0.002616
275932   -0.002616
Name: var_1_20_p20, Length: 275933, dtype: float64

In [27]:
ret_1 = features.cache['values']['ret_1']
ret_1

0              NaN
1         0.000000
2         0.158224
3         0.047628
4         0.000000
            ...   
275928   -0.003744
275929   -0.002253
275930    0.004875
275931    0.000000
275932    0.000000
Name: ret_1, Length: 275933, dtype: float64

In [35]:
ret_1.where(ret_1 < var_1_20_p20)

0              NaN
1              NaN
2              NaN
3              NaN
4              NaN
            ...   
275928   -0.003744
275929         NaN
275930         NaN
275931         NaN
275932         NaN
Name: ret_1, Length: 275933, dtype: float64

In [37]:
ret_1.groupby(df['symbol']).rolling(20).apply(lambda s: s[s <= s.quantile(0.2)].mean()).reset_index()

symbol        
ABAT    0              NaN
        1              NaN
        2              NaN
        3              NaN
        4              NaN
                    ...   
ZK      275928   -0.005390
        275929   -0.005390
        275930   -0.004013
        275931   -0.003552
        275932   -0.003552
Name: ret_1, Length: 275933, dtype: float64

In [38]:
VaR = ret_1.groupby(df['symbol']).rolling(20).quantile(0.2).reset_index(level=0, drop=True)
VaR

0              NaN
1              NaN
2              NaN
3              NaN
4              NaN
            ...   
275928   -0.003135
275929   -0.003135
275930   -0.002690
275931   -0.002616
275932   -0.002616
Name: ret_1, Length: 275933, dtype: float64

In [45]:
tail = ret_1.le(VaR)
ret_1.where(tail).groupby(df['symbol'], sort=False).rolling(20, min_periods=20).sum().reset_index(level=0, drop=True)

0        NaN
1        NaN
2        NaN
3        NaN
4        NaN
          ..
275928   NaN
275929   NaN
275930   NaN
275931   NaN
275932   NaN
Name: ret_1, Length: 275933, dtype: float64

In [50]:
VaR.tail(20)

275913   -0.010975
275914   -0.010975
275915   -0.010975
275916   -0.010357
275917   -0.008845
275918   -0.007487
275919   -0.007487
275920   -0.007487
275921   -0.006470
275922   -0.006470
275923   -0.006470
275924   -0.005147
275925   -0.004540
275926   -0.004540
275927   -0.003278
275928   -0.003135
275929   -0.003135
275930   -0.002690
275931   -0.002616
275932   -0.002616
Name: ret_1, dtype: float64

In [52]:
ret_1.tail(20)

275913    0.000745
275914   -0.002983
275915   -0.002617
275916    0.000000
275917   -0.000749
275918    0.000375
275919    0.003365
275920   -0.004864
275921    0.004491
275922   -0.000373
275923   -0.001121
275924    0.002241
275925   -0.002615
275926    0.000374
275927    0.000748
275928   -0.003744
275929   -0.002253
275930    0.004875
275931    0.000000
275932    0.000000
Name: ret_1, dtype: float64

In [55]:
q20 = ret_1.tail(20).quantile(0.2)
q20

-0.002615748521018056

In [60]:
VaR.tail(20)

275913   -0.010975
275914   -0.010975
275915   -0.010975
275916   -0.010357
275917   -0.008845
275918   -0.007487
275919   -0.007487
275920   -0.007487
275921   -0.006470
275922   -0.006470
275923   -0.006470
275924   -0.005147
275925   -0.004540
275926   -0.004540
275927   -0.003278
275928   -0.003135
275929   -0.003135
275930   -0.002690
275931   -0.002616
275932   -0.002616
Name: ret_1, dtype: float64

In [58]:
ret_1.le(VaR)

0         False
1         False
2         False
3         False
4         False
          ...  
275928     True
275929    False
275930    False
275931    False
275932    False
Name: ret_1, Length: 275933, dtype: bool

In [57]:
tail

0         False
1         False
2         False
3         False
4         False
          ...  
275928     True
275929    False
275930    False
275931    False
275932    False
Name: ret_1, Length: 275933, dtype: bool

In [47]:
ret_1.where(tail).tail(20)

275913         NaN
275914         NaN
275915         NaN
275916         NaN
275917         NaN
275918         NaN
275919         NaN
275920         NaN
275921         NaN
275922         NaN
275923         NaN
275924         NaN
275925         NaN
275926         NaN
275927         NaN
275928   -0.003744
275929         NaN
275930         NaN
275931         NaN
275932         NaN
Name: ret_1, dtype: float64

In [56]:
ret_1.tail(20).where(ret_1.tail(20).le(q20))

275913         NaN
275914   -0.002983
275915   -0.002617
275916         NaN
275917         NaN
275918         NaN
275919         NaN
275920   -0.004864
275921         NaN
275922         NaN
275923         NaN
275924         NaN
275925         NaN
275926         NaN
275927         NaN
275928   -0.003744
275929         NaN
275930         NaN
275931         NaN
275932         NaN
Name: ret_1, dtype: float64

In [62]:
ret_1.groupby(df['symbol']).rolling(20).apply(lambda s: s.values)

TypeError: only length-1 arrays can be converted to Python scalars

# Dates Features

In [48]:
df['ipoDate'] = np.where(df['ipoDate'].notna(), df['ipoDate'], df.groupby('symbol')['date'].transform("min"))

In [63]:
df['daysToIPO'] = (df['date'] - df['ipoDate']).dt.days.astype(float)
df['daysToFirstRec'] = (df['date'] - df.groupby('symbol')['date'].transform("min")).dt.days.astype(float)
df['daysToLastIncm'] = (df['date'] - df['incmFilingDate']).dt.days.astype(float)
df['daysToLastBal'] = (df['date'] - df['balFilingDate']).dt.days.astype(float)
df['daysToLastCf'] = (df['date'] - df['cfFilingDate']).dt.days.astype(float)

In [65]:
df.tail()

,symbol,date,adjOpen,adjHigh,adjLow,adjClose,adjVolume,rawClose,rawVolume,exchange,industry,sector,ipoDate,stockPrice,numberOfShares,marketCapitalization,minusCashAndCashEquivalents,addTotalDebt,evEnterpriseValue,price_tr20,dollarVolume,dollarVolume_tr20,turnOver,turnOver_tr20,lowLiquidity,incmFilingDate,incmAcceptedDate,revenue,costOfRevenue,grossProfit,researchAndDevelopmentExpenses,generalAndAdministrativeExpenses,sellingAndMarketingExpenses,sellingGeneralAndAdministrativeExpenses,otherExpenses,operatingExpenses,costAndExpenses,netInterestIncome,interestIncome,interestExpense,incmDepreciationAndAmortization,ebitda,ebit,nonOperatingIncomeExcludingInterest,operatingIncome,totalOtherIncomeExpensesNet,incomeBeforeTax,incomeTaxExpense,netIncomeFromContinuingOperations,netIncomeFromDiscontinuedOperations,otherAdjustmentsToNetIncome,incmNetIncome,netIncomeDeductions,bottomLineNetIncome,eps,epsDiluted,weightedAverageShsOut,weightedAverageShsOutDil,balFilingDate,balAcceptedDate,cashAndCashEquivalents,shortTermInvestments,cashAndShortTermInvestments,netReceivables,balAccountsReceivables,otherReceivables,balInventory,prepaids,otherCurrentAssets,totalCurrentAssets,propertyPlantEquipmentNet,goodwill,intangibleAssets,goodwillAndIntangibleAssets,longTermInvestments,taxAssets,otherNonCurrentAssets,totalNonCurrentAssets,otherAssets,totalAssets,totalPayables,accountPayables,otherPayables,accruedExpenses,shortTermDebt,capitalLeaseObligationsCurrent,taxPayables,deferredRevenue,otherCurrentLiabilities,totalCurrentLiabilities,longTermDebt,capitalLeaseObligationsNonCurrent,deferredRevenueNonCurrent,deferredTaxLiabilitiesNonCurrent,otherNonCurrentLiabilities,totalNonCurrentLiabilities,otherLiabilities,capitalLeaseObligations,totalLiabilities,treasuryStock,preferredStock,commonStock,retainedEarnings,additionalPaidInCapital,accumulatedOtherComprehensiveIncomeLoss,otherTotalStockholdersEquity,totalStockholdersEquity,totalEquity,minorityInterest,totalLiabilitiesAndTotalEquity,totalInvestments,totalDebt,netDebt,cfFilingDate,cfAcceptedDate,cfNetIncome,cfDepreciationAndAmortization,deferredIncomeTax,stockBasedCompensation,changeInWorkingCapital,cfAccountsReceivables,cfInventory,accountsPayables,otherWorkingCapital,otherNonCashItems,netCashProvidedByOperatingActivities,investmentsInPropertyPlantAndEquipment,acquisitionsNet,purchasesOfInvestments,salesMaturitiesOfInvestments,otherInvestingActivities,netCashProvidedByInvestingActivities,netDebtIssuance,longTermNetDebtIssuance,shortTermNetDebtIssuance,netStockIssuance,netCommonStockIssuance,commonStockIssuance,commonStockRepurchased,netPreferredStockIssuance,netDividendsPaid,commonDividendsPaid,preferredDividendsPaid,otherFinancingActivities,netCashProvidedByFinancingActivities,effectOfForexChangesOnCash,netChangeInCash,cashAtEndOfPeriod,cashAtBeginningOfPeriod,operatingCashFlow,capitalExpenditure,freeCashFlow,incomeTaxesPaid,interestPaid,marketCap,kmEnterpriseValue,evToSales,evToOperatingCashFlow,evToFreeCashFlow,evToEBITDA,netDebtToEBITDA,currentRatio,incomeQuality,grahamNetNet,taxBurden,interestBurden,workingCapital,investedCapital,returnOnAssets,operatingReturnOnAssets,returnOnTangibleAssets,returnOnEquity,returnOnInvestedCapital,returnOnCapitalEmployed,earningsYield,freeCashFlowYield,capexToOperatingCashFlow,capexToDepreciation,capexToRevenue,salesGeneralAndAdministrativeToRevenue,researchAndDevelopementToRevenue,stockBasedCompensationToRevenue,intangiblesToTotalAssets,averageReceivables,averagePayables,averageInventory,daysOfSalesOutstanding,daysOfPayablesOutstanding,daysOfInventoryOutstanding,operatingCycle,cashConversionCycle,freeCashFlowToEquity,freeCashFlowToFirm,tangibleAssetValue,netCurrentAssetValue,grahamNumber,daysToIPO,daysToFirstRec,daysToLastIncm,daysToLastBal,daysToLastCf
275928,ZK,2025-12-17,26.74,26.77,26.66,26.66,354521,26.66,354521.0,NYSE,Auto - Manufacturers,Consumer Cyclical,2024-05-10,216.97,257296840.0,5.582618e+10,7.021000e+09,3.162400e+10,8.042918e+10,26.7870,9451529.86,9.628350e

In [67]:
df.loc[df['daysToIPO'] != df['daysToFirstRec'], 'symbol'].drop_duplicates()

2476        ADUR
3359         AMT
15171       AZUR
15428        AZZ
24364       BMRA
43287     ECC-PD
44745       ETAO
48852       FLEX
55388       FLMN
56339       FRBA
62171       GMCR
65312       HCKT
73871        HSC
80084        IVC
85106       JKHY
91642       KBSX
91898       KNSA
109641       MDC
118751    MET-PA
123922      MSPR
125765       NOG
130480      NOTV
145563      OIIM
150455      OKSB
154009       OPK
169834       PPC
176370       PPG
182906       PRG
191556       PUK
197969      QCLS
198039       QSR
202364      RAVN
206952      REIS
209563     RILYZ
218902      SUIG
219009      T-PC
220485       THC
227021      TILE
233699       TRT
241630      VIVO
251188      VZLA
252176    WBS-PG
255385       WOR
261935      WSFS
269811         Y
Name: symbol, dtype: string

In [71]:
df.loc[df['symbol'].isin(['AZUR']), ['symbol', 'date', 'ipoDate']]

,symbol,date,ipoDate
15171,AZUR,2015-05-29,2015-01-30
15172,AZUR,2015-06-01,2015-01-30
15173,AZUR,2015-06-02,2015-01-30
15174,AZUR,2015-06-03,2015-01-30
15175,AZUR,2015-06-04,2015-01-30
...,...,...,...
15423,AZUR,2016-05-27,2015-01-30
15424,AZUR,2016-05-31,2015-01-30
15425,AZUR,2016-06-01,2015-01-30
15426,AZUR,2016-06-02,2015-01-30


# Row Number within Ticker (RN)

In [74]:
df['rn'] = df.groupby("symbol").cumcount() + 1
df[['symbol', 'date', 'rn']]

,symbol,date,rn
0,ABAT,2016-02-24,1
1,ABAT,2016-02-25,2
2,ABAT,2016-02-26,3
3,ABAT,2016-02-29,4
4,ABAT,2016-03-01,5
...,...,...,...
275928,ZK,2025-12-17,403
275929,ZK,2025-12-18,404
275930,ZK,2025-12-19,405
275931,ZK,2025-12-22,406


In [7]:
import base64
import struct
import math
from typing import Tuple, Union

Atom = Union[int, float, bool, str]

TYPE_INT   = b"i"
TYPE_FLOAT = b"f"
TYPE_BOOL  = b"b"
TYPE_STR   = b"s"

def tuple_to_string(t: Tuple[Atom, ...]) -> str:
    buf = bytearray()

    for x in t:
        if isinstance(x, bool):
            buf.extend(TYPE_BOOL)
            buf.extend(b"\x01" if x else b"\x00")

        elif isinstance(x, int):
            b = str(x).encode("utf-8")
            buf.extend(TYPE_INT)
            buf.extend(struct.pack(">I", len(b)))
            buf.extend(b)

        elif isinstance(x, float):
            buf.extend(TYPE_FLOAT)
            buf.extend(struct.pack(">d", x))

        elif isinstance(x, str):
            b = x.encode("utf-8")
            buf.extend(TYPE_STR)
            buf.extend(struct.pack(">I", len(b)))
            buf.extend(b)

        else:
            raise TypeError(f"Unsupported type: {type(x)}")

    return base64.urlsafe_b64encode(buf).decode("ascii")


def string_to_tuple(s: str) -> Tuple[Atom, ...]:
    try:
        data = base64.urlsafe_b64decode(s.encode("ascii"))
    except Exception as e:
        raise ValueError("Invalid encoded string") from e

    i = 0
    out = []

    while i < len(data):
        tag = data[i:i+1]
        i += 1

        if tag == TYPE_BOOL:
            out.append(data[i] == 1)
            i += 1

        elif tag == TYPE_INT:
            n = struct.unpack(">I", data[i:i+4])[0]
            i += 4
            out.append(int(data[i:i+n].decode("utf-8")))
            i += n

        elif tag == TYPE_FLOAT:
            out.append(struct.unpack(">d", data[i:i+8])[0])
            i += 8

        elif tag == TYPE_STR:
            n = struct.unpack(">I", data[i:i+4])[0]
            i += 4
            out.append(data[i:i+n].decode("utf-8"))
            i += n

        else:
            raise ValueError(f"Unknown type tag: {tag}")

    return tuple(out)


In [8]:
tuple_to_string(('ret', 10))

'cwAAAANyZXRpAAAAAjEw'

In [9]:
string_to_tuple('cwAAAANyZXRpAAAAAjEw')

('ret', 10)

In [10]:
tuple_to_string(('ret', 12))

'cwAAAANyZXRpAAAAAjEy'

In [12]:
string_to_tuple('cwAAAANyZXRpAAAAAjEz')

('ret', 13)

In [14]:
import numpy as np


arr = np.random.rand(30)
arr = np.array(range(30))
arr

array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
       17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29])

In [15]:
from numpy.lib.stride_tricks import sliding_window_view


sliding_window_view(arr, 5)

array([[ 0,  1,  2,  3,  4],
       [ 1,  2,  3,  4,  5],
       [ 2,  3,  4,  5,  6],
       [ 3,  4,  5,  6,  7],
       [ 4,  5,  6,  7,  8],
       [ 5,  6,  7,  8,  9],
       [ 6,  7,  8,  9, 10],
       [ 7,  8,  9, 10, 11],
       [ 8,  9, 10, 11, 12],
       [ 9, 10, 11, 12, 13],
       [10, 11, 12, 13, 14],
       [11, 12, 13, 14, 15],
       [12, 13, 14, 15, 16],
       [13, 14, 15, 16, 17],
       [14, 15, 16, 17, 18],
       [15, 16, 17, 18, 19],
       [16, 17, 18, 19, 20],
       [17, 18, 19, 20, 21],
       [18, 19, 20, 21, 22],
       [19, 20, 21, 22, 23],
       [20, 21, 22, 23, 24],
       [21, 22, 23, 24, 25],
       [22, 23, 24, 25, 26],
       [23, 24, 25, 26, 27],
       [24, 25, 26, 27, 28],
       [25, 26, 27, 28, 29]])

In [6]:
return_1d = features.cache['values']['ret_1']
returns = pd.concat([df[['symbol', 'date']], return_1d], axis=1)
returns

,symbol,date,ret_1
0,ABAT,2016-02-24,NaN
1,ABAT,2016-02-25,0.000000
2,ABAT,2016-02-26,0.158224
3,ABAT,2016-02-29,0.047628
4,ABAT,2016-03-01,0.000000
...,...,...,...
275928,ZK,2025-12-17,-0.003744
275929,ZK,2025-12-18,-0.002253
275930,ZK,2025-12-19,0.004875
275931,ZK,2025-12-22,0.000000


In [7]:
return_2d = pd.pivot(returns, columns='symbol', index='date', values='ret_1')
return_2d

symbol,ABAT,ADUR,AEVA-WT,AMT,AQNB,ASPI,ATHA,AVRO,AZUR,AZZ,BDCIW,BLBX,BMRA,BTOG,CAI,CMRE-PB,CSII,DC,DDL,DSP,ECC-PD,EMBKW,ETAO,EVOP,EVTV,FLEX,FLMN,FRBA,GAMB,GMCR,HCKT,HEXO,HOVRW,HSC,HTLM,IVC,JEM,JKHY,KBSX,KNSA,KRNY,LAB,LRN,LSBPW,LTHM,MBC,MDC,MDWD,MEHA,MET-PA,MSPR,NET,NOG,NOTV,NTRA,NUVA,OFSSH,OIIM,OKSB,OPK,PEB-PE,PEB-PF,PFS,PPC,PPG,PRG,PSA-PG,PUK,QCLS,QSR,RAPT,RAVN,REIS,RILYZ,RMR,SEMR,SIVBP,SPRO,STBX,SUIG,T-PC,THC,TILE,TRCA-WT,TRT,TXO,UP-WT,VECT,VIVO,VLDRW,VRSK,VZLA,WBS-PG,WOR,WPCA-WT,WSFS,XPEV,Y,ZAPP,ZK
date,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2000-01-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2000-01-04,NaN,NaN,NaN,-0.010685,NaN,NaN,NaN,NaN,NaN,-0.004357,NaN,NaN,-0.085230,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.097389,NaN,NaN,NaN,NaN,-0.056805,NaN,NaN,-0.014001,NaN,NaN,NaN,-0.085180,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.008658,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,-0.015484,NaN,NaN,NaN,-0.046884,-0.035047,-0.004291,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.007826,-0.144630,NaN,0.036258,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.015103,NaN,-0.029081,NaN,NaN,NaN,NaN
2000-01-05,NaN,NaN,NaN,0.029908,NaN,NaN,NaN,NaN,NaN,-0.022076,NaN,NaN,0.065428,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.016700,NaN,NaN,NaN,NaN,0.101414,NaN,NaN,0.002167,NaN,NaN,NaN,0.020642,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.008658,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,0.015873,0.036923,-1.381392,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.015591,0.040005,NaN,0.032120,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.019089,NaN,0.003273,NaN,NaN,NaN,NaN
2000-01-06,NaN,NaN,NaN,-0.010939,NaN,NaN,NaN,NaN,NaN,-0.008969,NaN,NaN,-0.040822,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.010838,NaN,NaN,NaN,NaN,-0.067682,NaN,NaN,0.001082,NaN,NaN,NaN,-0.009725,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.002878,NaN,NaN,NaN,NaN,NaN,NaN,0.066809,NaN,NaN,NaN,NaN,NaN,-0.025018,NaN,NaN,NaN,0.015625,0.047003,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.059896,-0.052842,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.014815,NaN,0.006515,NaN,NaN,NaN,NaN
2000-01-07,NaN,NaN,NaN,0.053971,NaN,NaN,NaN,NaN,NaN,-0.009050,NaN,NaN,0.040822,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.040332,NaN,NaN,NaN,NaN,-0.007811,NaN,NaN,0.006466,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.014306,NaN,NaN,NaN,NaN,NaN,NaN,-0.020619,NaN,NaN,NaN,NaN,NaN,0.049425,NaN,NaN,NaN,0.039891,0.017731,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.070323,0.052842,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.047876,NaN,-0.006515,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-12-19,0.051691,-0.032571,NaN,-0.010845,-0.000782,0.048872,0.036527,NaN,NaN,-0.001205,NaN,0.008615,-0.017168,0.020514,0.002867,-0.007695,NaN,0.019081,-0.027186,-0.012674,0.001532,NaN,NaN,NaN,-0.226599,0.018857,NaN,-0.015625,-0.026770,NaN,-0.003526,NaN,-0.035083,NaN,-0.136904,NaN,0.006920,0.000651,0.014493,0.027939,-0.006333,-0.018238,0.004550,NaN,NaN,-0.024671,NaN,-0.003722,0.099563,-0.002706,-0.262364,0.009499,-0.011998,0.032710,0.031621,NaN,0.0,NaN,NaN,0.000000,0.000000,0.011050,-0.013302,0.002539,-0.002237,-0.022777,0.001474,

In [8]:
return_2d.rolling(window=50, min_periods=50).quantile(0.05)

symbol,ABAT,ADUR,AEVA-WT,AMT,AQNB,ASPI,ATHA,AVRO,AZUR,AZZ,BDCIW,BLBX,BMRA,BTOG,CAI,CMRE-PB,CSII,DC,DDL,DSP,ECC-PD,EMBKW,ETAO,EVOP,EVTV,FLEX,FLMN,FRBA,GAMB,GMCR,HCKT,HEXO,HOVRW,HSC,HTLM,IVC,JEM,JKHY,KBSX,KNSA,KRNY,LAB,LRN,LSBPW,LTHM,MBC,MDC,MDWD,MEHA,MET-PA,MSPR,NET,NOG,NOTV,NTRA,NUVA,OFSSH,OIIM,OKSB,OPK,PEB-PE,PEB-PF,PFS,PPC,PPG,PRG,PSA-PG,PUK,QCLS,QSR,RAPT,RAVN,REIS,RILYZ,RMR,SEMR,SIVBP,SPRO,STBX,SUIG,T-PC,THC,TILE,TRCA-WT,TRT,TXO,UP-WT,VECT,VIVO,VLDRW,VRSK,VZLA,WBS-PG,WOR,WPCA-WT,WSFS,XPEV,Y,ZAPP,ZK
date,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2000-01-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2000-01-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2000-01-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2000-01-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2000-01-07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-12-19,-0.183818,-0.073347,NaN,-0.022153,-0.004273,-0.111274,-0.045786,NaN,NaN,-0.024305,NaN,-0.138463,-0.050124,-0.115300,-0.046247,-0.010184,NaN,-0.051913,-0.038578,-0.051132,-0.007254,NaN,NaN,NaN,-0.168318,-0.048703,NaN,-0.017130,-0.044849,NaN,-0.021856,NaN,-0.129373,NaN,-0.067779,NaN,-0.109387,-0.013555,-0.040358,-0.026414,-0.024927,-0.063066,-0.044261,NaN,NaN,-0.034443,NaN,-0.037993,NaN,-0.010464,-0.345616,-0.040365,-0.040294,-0.083616,-0.020855,NaN,-0.006841,NaN,NaN,-0.033597,-0.017071,-0.014192,-0.030691,-0.024911,-0.019617,-0.036518,-0.011699,-0.015872,-0.148707,-0.022689,-0.087860,NaN,NaN,-0.044072,-0.022006,-0.044291,NaN,-0.056887,NaN,-0.119632,-0.009882,-0.034201,-0.022764,NaN,-0.046578,-0.033595,NaN,NaN,NaN,NaN,-0.02204,-0.075591,-0.011852,-0.03063,NaN,-0.027081,-0.075168,NaN,NaN,-0.029596
2025-12-22,-0.183818,-0.073347,NaN,-0.022153,-0.004273,-0.111274,-0.045786,NaN,NaN,-0.024305,NaN,-0.138463,-0.05

In [66]:
r = return_2d.iloc[-50:, 0]
r

date
2025-10-16   -0.456464
2025-10-17   -0.156210
2025-10-20    0.326109
2025-10-21   -0.096799
2025-10-22   -0.026580
2025-10-23   -0.055377
2025-10-24    0.062089
2025-10-27   -0.103875
2025-10-28   -0.047492
2025-10-29    0.040043
2025-10-30   -0.045897
2025-10-31    0.003906
2025-11-03   -0.192904
2025-11-04   -0.073563
2025-11-05    0.012642
2025-11-06   -0.075655
2025-11-07    0.182773
2025-11-10   -0.034447
2025-11-11   -0.052770
2025-11-12    0.068993
2025-11-13   -0.172713
2025-11-14   -0.016529
2025-11-17   -0.066021
2025-11-18    0.090714
2025-11-19    0.085655
2025-11-20   -0.135666
2025-11-21   -0.040703
2025-11-24    0.043548
2025-11-25    0.011300
2025-11-26    0.005602
2025-11-28    0.043723
2025-12-01   -0.101210
2025-12-02    0.082316
2025-12-03    0.021564
2025-12-04    0.110945
2025-12-05   -0.009592
2025-12-08    0.019094
2025-12-09    0.004717
2025-12-10   -0.058128
2025-12-11    0.062823
2025-12-12   -0.043069
2025-12-15   -0.039906
2025-12-16    0.020152
2025-1

In [67]:
q005 = r.quantile(0.05)
q005

-0.1652866694836458

In [68]:
r[r<q005]

date
2025-10-16   -0.456464
2025-11-03   -0.192904
2025-11-13   -0.172713
Name: ABAT, dtype: float64

In [53]:
v[-1, :]

array([ 0.04372281, -0.1012099 ,  0.08231595,  0.02156418,  0.11094489,
       -0.0095924 ,  0.01909366,  0.00471699, -0.05812774,  0.06282259,
       -0.04306886, -0.03990554,  0.02015182, -0.09680756,  0.03509132,
        0.05169109,  0.04674898, -0.0169701 , -0.02977888, -0.03851567])

In [9]:
qval = return_2d.rolling(window=50, min_periods=50).quantile(0.05).to_numpy()
qval = qval[(50 - 1):, :]
qval

array([[        nan,         nan,         nan, ...,         nan,
                nan,         nan],
       [        nan,         nan,         nan, ...,         nan,
                nan,         nan],
       [        nan,         nan,         nan, ...,         nan,
                nan,         nan],
       ...,
       [-0.18381773, -0.07334714,         nan, ...,         nan,
                nan, -0.02571606],
       [-0.18381773, -0.07334714,         nan, ...,         nan,
                nan,         nan],
       [-0.16528667, -0.07334714,         nan, ...,         nan,
                nan,         nan]])

In [10]:
qval.shape

(6487, 100)

In [11]:
return_2d_arr = return_2d.to_numpy()
return_2d_arr

array([[        nan,         nan,         nan, ...,         nan,
                nan,         nan],
       [        nan,         nan,         nan, ...,         nan,
                nan,         nan],
       [        nan,         nan,         nan, ...,         nan,
                nan,         nan],
       ...,
       [-0.0169701 , -0.01288124,         nan, ...,         nan,
                nan,  0.        ],
       [-0.02977888, -0.02804741,         nan, ...,         nan,
                nan,         nan],
       [-0.03851567,  0.00796817,         nan, ...,         nan,
                nan,         nan]])

In [12]:
return_2d_arr.shape

(6536, 100)

In [14]:
return_2d_arr[:, 2]

array([nan, nan, nan, ..., nan, nan, nan])

In [15]:
(~np.isnan(return_2d_arr[:, 2])).sum()

580

In [16]:
(~np.isnan(return_2d_arr)).sum(axis=0)

array([2475,  301,  580, 6535, 1656,  783, 1324, 1509,  256, 6535,   34,
       2364, 6535, 1600,  132, 3120, 4239,  935, 1129, 1225, 1027,  429,
        741, 1218, 2145, 6535,  950, 4718, 1112, 3140, 6535, 1572,  449,
       5901,  310, 4881,  139, 6535,  255, 1908, 5243, 3741, 4537,  212,
       1332,  763, 6111, 2961,   35, 5170,  261, 1580, 4714, 6535, 2638,
       4863, 1043, 4891, 3553, 6535, 1738, 1775, 5773, 6535, 6535, 6535,
       2113, 6412,   69, 2777, 1546, 4587, 2610, 2063, 2523, 1196,  813,
       2047,  691,  106, 1475, 6535, 6535,  141, 6533,  731,   98,  565,
       4880,  595, 4080,  987, 3208, 6535,   13, 6535, 1339, 4810,  903,
        406])

In [17]:
reutrn_tk1 = return_2d_arr[:, 0]
reutrn_tk1

array([        nan,         nan,         nan, ..., -0.0169701 ,
       -0.02977888, -0.03851567])

In [18]:
reutrn_tk1_r20 = sliding_window_view(reutrn_tk1, 20)
reutrn_tk1_r20

array([[        nan,         nan,         nan, ...,         nan,
                nan,         nan],
       [        nan,         nan,         nan, ...,         nan,
                nan,         nan],
       [        nan,         nan,         nan, ...,         nan,
                nan,         nan],
       ...,
       [ 0.01129956,  0.00560226,  0.04372281, ...,  0.05169109,
         0.04674898, -0.0169701 ],
       [ 0.00560226,  0.04372281, -0.1012099 , ...,  0.04674898,
        -0.0169701 , -0.02977888],
       [ 0.04372281, -0.1012099 ,  0.08231595, ..., -0.0169701 ,
        -0.02977888, -0.03851567]])

In [19]:
reutrn_tk1_r20[:, 0]

array([       nan,        nan,        nan, ..., 0.01129956, 0.00560226,
       0.04372281])

In [16]:
return_r50 = sliding_window_view(return_2d_arr, 50, axis=0)
return_r50.shape

(6487, 100, 50)

In [74]:
v = return_r50[:, 0, :]
v

array([[        nan,         nan,         nan, ...,         nan,
                nan,         nan],
       [        nan,         nan,         nan, ...,         nan,
                nan,         nan],
       [        nan,         nan,         nan, ...,         nan,
                nan,         nan],
       ...,
       [ 0.20607904, -0.23580054, -0.45646441, ...,  0.05169109,
         0.04674898, -0.0169701 ],
       [-0.23580054, -0.45646441, -0.15621041, ...,  0.04674898,
        -0.0169701 , -0.02977888],
       [-0.45646441, -0.15621041,  0.32610945, ..., -0.0169701 ,
        -0.02977888, -0.03851567]])

In [38]:
x = np.isnan(v).mean(axis=1)
x

array([1., 1., 1., ..., 0., 0., 0.])

In [41]:
x[(x<1) & (x>0)]

array([0.95, 0.9 , 0.85, 0.8 , 0.75, 0.7 , 0.65, 0.6 , 0.55, 0.5 , 0.45,
       0.4 , 0.35, 0.3 , 0.25, 0.2 , 0.15, 0.1 , 0.05])

In [43]:
q = np.nanquantile(return_r20, 0.05, axis=2)
q.shape

/opt/conda/lib/python3.11/site-packages/numpy/lib/nanfunctions.py:1563: RuntimeWarning: All-NaN slice encountered
  return function_base._ureduce(a,


(6517, 100)

In [48]:
q[:30, 0]

array([nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
       nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
       nan, nan, nan, nan])

In [17]:
qval.shape, return_r50.shape

((6487, 100), (6487, 100, 50))

In [80]:
return_r50[-1, 0, :]

array([-0.45646441, -0.15621041,  0.32610945, -0.09679938, -0.02657964,
       -0.05537747,  0.0620889 , -0.10387518, -0.04749231,  0.04004348,
       -0.04589716,  0.00390625, -0.19290367, -0.07356257,  0.01264239,
       -0.07565536,  0.18277313, -0.03444657, -0.05277004,  0.06899287,
       -0.1727127 , -0.0165293 , -0.0660211 ,  0.09071371,  0.08565544,
       -0.13566587, -0.04070329,  0.04354825,  0.01129956,  0.00560226,
        0.04372281, -0.1012099 ,  0.08231595,  0.02156418,  0.11094489,
       -0.0095924 ,  0.01909366,  0.00471699, -0.05812774,  0.06282259,
       -0.04306886, -0.03990554,  0.02015182, -0.09680756,  0.03509132,
        0.05169109,  0.04674898, -0.0169701 , -0.02977888, -0.03851567])

In [83]:
qval[-1, 0]

-0.1652866694836458

In [84]:
return_r50[-1, 0, :] < qval[-1, 0]

array([ True, False, False, False, False, False, False, False, False,
       False, False, False,  True, False, False, False, False, False,
       False, False,  True, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False])

In [96]:
qval[:, :, None].shape

(6487, 100, 1)

In [18]:
mask = return_r50 < qval[:, :, None]
mask.shape

(6487, 100, 50)

In [19]:
filtered = np.where(mask, return_r50, np.nan)
filtered.shape

(6487, 100, 50)

In [20]:
filtered[-1, 0, :]

array([-0.45646441,         nan,         nan,         nan,         nan,
               nan,         nan,         nan,         nan,         nan,
               nan,         nan, -0.19290367,         nan,         nan,
               nan,         nan,         nan,         nan,         nan,
       -0.1727127 ,         nan,         nan,         nan,         nan,
               nan,         nan,         nan,         nan,         nan,
               nan,         nan,         nan,         nan,         nan,
               nan,         nan,         nan,         nan,         nan,
               nan,         nan,         nan,         nan,         nan,
               nan,         nan,         nan,         nan,         nan])

In [21]:
out = np.nanmean(filtered, axis=2)
out.shape

/tmp/ipykernel_156/4239774566.py:1: RuntimeWarning: Mean of empty slice
  out = np.nanmean(filtered, axis=2)


(6487, 100)

In [22]:
out[:, 0]

array([        nan,         nan,         nan, ..., -0.29505621,
       -0.29505621, -0.27402693])

In [111]:
return_2d.iloc[(50 - 1):].index

DatetimeIndex(['2000-03-14', '2000-03-15', '2000-03-16', '2000-03-17',
               '2000-03-20', '2000-03-21', '2000-03-22', '2000-03-23',
               '2000-03-24', '2000-03-27',
               ...
               '2025-12-12', '2025-12-15', '2025-12-16', '2025-12-17',
               '2025-12-18', '2025-12-19', '2025-12-22', '2025-12-23',
               '2025-12-24', '2025-12-26'],
              dtype='datetime64[ns]', name='date', length=6487, freq=None)

In [112]:
return_2d.columns

Index(['ABAT', 'ADUR', 'AEVA-WT', 'AMT', 'AQNB', 'ASPI', 'ATHA', 'AVRO',
       'AZUR', 'AZZ', 'BDCIW', 'BLBX', 'BMRA', 'BTOG', 'CAI', 'CMRE-PB',
       'CSII', 'DC', 'DDL', 'DSP', 'ECC-PD', 'EMBKW', 'ETAO', 'EVOP', 'EVTV',
       'FLEX', 'FLMN', 'FRBA', 'GAMB', 'GMCR', 'HCKT', 'HEXO', 'HOVRW', 'HSC',
       'HTLM', 'IVC', 'JEM', 'JKHY', 'KBSX', 'KNSA', 'KRNY', 'LAB', 'LRN',
       'LSBPW', 'LTHM', 'MBC', 'MDC', 'MDWD', 'MEHA', 'MET-PA', 'MSPR', 'NET',
       'NOG', 'NOTV', 'NTRA', 'NUVA', 'OFSSH', 'OIIM', 'OKSB', 'OPK', 'PEB-PE',
       'PEB-PF', 'PFS', 'PPC', 'PPG', 'PRG', 'PSA-PG', 'PUK', 'QCLS', 'QSR',
       'RAPT', 'RAVN', 'REIS', 'RILYZ', 'RMR', 'SEMR', 'SIVBP', 'SPRO', 'STBX',
       'SUIG', 'T-PC', 'THC', 'TILE', 'TRCA-WT', 'TRT', 'TXO', 'UP-WT', 'VECT',
       'VIVO', 'VLDRW', 'VRSK', 'VZLA', 'WBS-PG', 'WOR', 'WPCA-WT', 'WSFS',
       'XPEV', 'Y', 'ZAPP', 'ZK'],
      dtype='string', name='symbol')

In [25]:
out[-1, :]

array([-0.27402693, -0.09131529,         nan, -0.02980993, -0.00855507,
       -0.12220509, -0.07165392,         nan,         nan, -0.03127325,
               nan, -0.18159469, -0.07597113, -0.12614575, -0.09501122,
       -0.01721939,         nan, -0.10329693, -0.05790592, -0.05416481,
       -0.00809606,         nan,         nan,         nan, -0.27455641,
       -0.05562551,         nan, -0.03974962, -0.12997807,         nan,
       -0.0253943 ,         nan, -0.16235322,         nan, -0.11789273,
               nan, -0.13659595, -0.02102535, -0.05272375, -0.04024784,
       -0.03343862, -0.08939461, -0.31096338,         nan,         nan,
       -0.07604034,         nan, -0.05077039,         nan, -0.01409097,
               nan, -0.05299001, -0.04474301, -0.19069626, -0.02823731,
               nan,         nan,         nan,         nan, -0.04716247,
       -0.02188241, -0.02381972, -0.04308014, -0.03101837, -0.03695291,
       -0.04542718, -0.01351186, -0.01951245, -0.18260575, -0.02

In [28]:
out_df = pd.DataFrame(out, index=return_2d.iloc[(50 - 1):].index, columns=return_2d.columns)
out_df = out_df.reindex(return_2d.index)
out_df

symbol,ABAT,ADUR,AEVA-WT,AMT,AQNB,ASPI,ATHA,AVRO,AZUR,AZZ,BDCIW,BLBX,BMRA,BTOG,CAI,CMRE-PB,CSII,DC,DDL,DSP,ECC-PD,EMBKW,ETAO,EVOP,EVTV,FLEX,FLMN,FRBA,GAMB,GMCR,HCKT,HEXO,HOVRW,HSC,HTLM,IVC,JEM,JKHY,KBSX,KNSA,KRNY,LAB,LRN,LSBPW,LTHM,MBC,MDC,MDWD,MEHA,MET-PA,MSPR,NET,NOG,NOTV,NTRA,NUVA,OFSSH,OIIM,OKSB,OPK,PEB-PE,PEB-PF,PFS,PPC,PPG,PRG,PSA-PG,PUK,QCLS,QSR,RAPT,RAVN,REIS,RILYZ,RMR,SEMR,SIVBP,SPRO,STBX,SUIG,T-PC,THC,TILE,TRCA-WT,TRT,TXO,UP-WT,VECT,VIVO,VLDRW,VRSK,VZLA,WBS-PG,WOR,WPCA-WT,WSFS,XPEV,Y,ZAPP,ZK
date,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2000-01-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2000-01-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2000-01-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2000-01-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2000-01-07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-12-19,-0.295056,-0.091315,NaN,-0.02981,-0.008555,-0.122205,-0.071654,NaN,NaN,-0.031273,NaN,-0.181595,-0.075971,-0.141870,-0.095011,-0.016062,NaN,-0.103297,-0.061194,-0.056777,-0.008968,NaN,NaN,NaN,-0.274556,-0.058309,NaN,-0.033456,-0.129978,NaN,-0.029847,NaN,-0.205939,NaN,-0.108752,NaN,-0.136596,-0.021025,-0.052724,-0.040248,-0.035552,-0.073306,-0.310963,NaN,NaN,-0.076883,NaN,-0.05077,NaN,-0.014091,-0.487611,-0.05299,-0.055000,-0.190696,-0.029192,NaN,-0.009515,NaN,NaN,-0.047162,-0.021882,-0.022907,-0.050211,-0.032294,-0.036953,-0.045427,-0.013824,-0.019512,-0.182606,-0.025944,-0.109141,NaN,NaN,-0.056848,-0.032393,-0.053547,NaN,-0.068539,NaN,-0.142182,-0.012732,-0.049194,-0.04299,NaN,-0.073387,-0.043136,NaN,NaN,NaN,NaN,-0.066735,-0.121724,-0.022216,-0.039823,NaN,-0.045425,-0.092480,NaN,NaN,-0.036720
2025-12-22,-0.295056,-0.091315,NaN,-0.02981,-0.008555,-0.122205,-0.071654,NaN,NaN,-0.031273,NaN,-0.181595,-0.07597

In [31]:
out_long_df = out_df.reset_index().melt(
    id_vars=out_df.index.name,
    var_name=out_df.columns.name,
    value_name='es_p05_50'
)

out_long_df

,date,symbol,es_p05_50
0,2000-01-03,ABAT,NaN
1,2000-01-04,ABAT,NaN
2,2000-01-05,ABAT,NaN
3,2000-01-06,ABAT,NaN
4,2000-01-07,ABAT,NaN
...,...,...,...
653595,2025-12-19,ZK,-0.036720
653596,2025-12-22,ZK,-0.032185
653597,2025-12-23,ZK,-0.032185
653598,2025-12-24,ZK,NaN


In [32]:
returns.merge(out_long_df, how='left', on=['symbol', 'date'])

,symbol,date,ret_1,es_p05_50
0,ABAT,2016-02-24,NaN,NaN
1,ABAT,2016-02-25,0.000000,NaN
2,ABAT,2016-02-26,0.158224,NaN
3,ABAT,2016-02-29,0.047628,NaN
4,ABAT,2016-03-01,0.000000,NaN
...,...,...,...,...
275928,ZK,2025-12-17,-0.003744,-0.036720
275929,ZK,2025-12-18,-0.002253,-0.036720
275930,ZK,2025-12-19,0.004875,-0.036720
275931,ZK,2025-12-22,0.000000,-0.032185


In [33]:
returns

,symbol,date,ret_1
0,ABAT,2016-02-24,NaN
1,ABAT,2016-02-25,0.000000
2,ABAT,2016-02-26,0.158224
3,ABAT,2016-02-29,0.047628
4,ABAT,2016-03-01,0.000000
...,...,...,...
275928,ZK,2025-12-17,-0.003744
275929,ZK,2025-12-18,-0.002253
275930,ZK,2025-12-19,0.004875
275931,ZK,2025-12-22,0.000000


In [29]:
from numpy.lib.stride_tricks import sliding_window_view

r = features.cache['values']['ret_1']
ret_long_df = pd.concat([df[['symbol', 'date']], r], axis=1)
ret_wide_df = pd.pivot(ret_long_df, columns='symbol', index='date', values='ret_1')

q_arr = ret_wide_df.rolling(window=50, min_periods=50).quantile(0.05).to_numpy()
q_arr = q_arr[(50 - 1):, :]

ret_arr = ret_wide_df.to_numpy()
ret_arr_r50 = sliding_window_view(ret_arr, 50, axis=0)

mask = ret_arr_r50 < q_arr[:, :, None]
# filtered = np.where(mask, ret_arr_r50, np.nan)
# out = np.nanmean(filtered, axis=2)
sum_ = np.sum(np.where(mask, ret_arr_r50, 0.0), axis=2)
cnt = np.sum(mask, axis=2)
out = np.divide(sum_, cnt, out=np.full_like(sum_, np.nan), where=cnt > 0)

out_wide_df = pd.DataFrame(out, index=ret_wide_df.iloc[(50 - 1):].index, columns=ret_wide_df.columns)
out_wide_df = out_wide_df.reindex(ret_wide_df.index)
out_long_df = out_wide_df.reset_index().melt(
    id_vars=out_wide_df.index.name,
    var_name=out_wide_df.columns.name,
    value_name='es_p05_50'
)
ret_long_df2 = ret_long_df.merge(out_long_df, how='left', on=['symbol', 'date'])

ret_long_df2.shape

(275933, 4)

In [53]:
ret_long_df2

,symbol,date,ret_1,es_p05_50
0,ABAT,2016-02-24,NaN,NaN
1,ABAT,2016-02-25,0.000000,NaN
2,ABAT,2016-02-26,0.158224,NaN
3,ABAT,2016-02-29,0.047628,NaN
4,ABAT,2016-03-01,0.000000,NaN
...,...,...,...,...
275928,ZK,2025-12-17,-0.003744,-0.036720
275929,ZK,2025-12-18,-0.002253,-0.036720
275930,ZK,2025-12-19,0.004875,-0.036720
275931,ZK,2025-12-22,0.000000,-0.032185


In [30]:
r = ret_long_df2.iloc[-52:-2, 2]
r.shape

(50,)

In [31]:
r[r < r.quantile(0.05)].mean()

-0.03672045108435763

In [32]:
ret_arr_r50.shape

(6487, 100, 50)

In [26]:
sum_ = np.sum(np.where(mask, ret_arr_r50, 0.0), axis=2)
cnt = np.sum(mask, axis=2)
out = np.divide(sum_, cnt, out=np.full_like(sum_, np.nan), where=cnt > 0)
out

array([[        nan,         nan,         nan, ...,         nan,
                nan,         nan],
       [        nan,         nan,         nan, ...,         nan,
                nan,         nan],
       [        nan,         nan,         nan, ...,         nan,
                nan,         nan],
       ...,
       [-0.29505621, -0.09131529,         nan, ...,         nan,
                nan, -0.0321846 ],
       [-0.29505621, -0.09131529,         nan, ...,         nan,
                nan,         nan],
       [-0.27402693, -0.09131529,         nan, ...,         nan,
                nan,         nan]])

In [54]:
arr = np.array([1.2, 0.9, 0.001, 2.2, 0, -1.1, -0.2])
eps = 0.1

arr

array([ 1.2e+00,  9.0e-01,  1.0e-03,  2.2e+00,  0.0e+00, -1.1e+00,
       -2.0e-01])

In [55]:
np.maximum(arr, eps)

array([1.2, 0.9, 0.1, 2.2, 0.1, 0.1, 0.1])

# Feature Primitives

In [4]:
from __future__ import annotations
from dataclasses import dataclass
from typing import Any, Callable, Dict, Hashable, Optional, List, Tuple
import numpy as np
import pandas as pd
import inspect


# ---------- Spec ----------
PostOp = Tuple[str, Dict[str, Any]]   # ("clip", {"low": -5, "high": 5})


@dataclass(frozen=True)
class FeatureSpec:
    name: str
    primitive: str
    inputs: Dict[str, Any]            # values can be: df col name, other spec name, literal, Series
    params: Dict[str, Any] = None
    post: Optional[List[PostOp]] = None
    publish: bool = True


# ---------- Primitive helpers ----------
def _gb(df: pd.DataFrame, sym_col: str):
    return df.groupby(sym_col, sort=False)


def prim_ts_return(df, sym_col, price: pd.Series, lookback: int) -> pd.Series:
    g = price.groupby(df[sym_col], sort=False)
    out = np.log1p(g.pct_change(lookback, fill_method=None))
    out.name = None
    return out


def prim_ts_mean(df, sym_col, x: pd.Series, window: int, min_periods: Optional[int] = None) -> pd.Series:
    if min_periods is None:
        min_periods = window
    g = x.groupby(df[sym_col], sort=False)
    out = g.transform(lambda s: s.rolling(window, min_periods=min_periods).mean())
    out.name = None
    return out


def prim_ts_std(df, sym_col, x: pd.Series, window: int, min_periods: Optional[int] = None) -> pd.Series:
    if min_periods is None:
        min_periods = window
    g = x.groupby(df[sym_col], sort=False)
    out = g.transform(lambda s: s.rolling(window, min_periods=min_periods).std())
    out.name = None
    return out


def prim_ts_max(df, sym_col, x: pd.Series, window: int, min_periods: Optional[int] = None) -> pd.Series:
    if min_periods is None:
        min_periods = window
    g = x.groupby(df[sym_col], sort=False)
    out = g.transform(lambda s: s.rolling(window, min_periods=min_periods).max())
    out.name = None
    return out


def prim_ts_min(df, sym_col, x: pd.Series, window: int, min_periods: Optional[int] = None) -> pd.Series:
    if min_periods is None:
        min_periods = window
    g = x.groupby(df[sym_col], sort=False)
    out = g.transform(lambda s: s.rolling(window, min_periods=min_periods).min())
    out.name = None
    return out


def prim_ts_ewm_mean(df, sym_col, x: pd.Series, window: int, min_periods: Optional[int] = None) -> pd.Series:
    if min_periods is None:
        min_periods = window
    g = x.groupby(df[sym_col], sort=False)
    out = g.transform(lambda s: s.ewm(alpha=1/window, adjust=False, min_periods=min_periods).mean())
    out.name = None
    return out


def prim_ts_ewm_span_mean(df, sym_col, x: pd.Series, span: int, min_periods: Optional[int] = None) -> pd.Series:
    if min_periods is None:
        min_periods = span
    g = x.groupby(df[sym_col], sort=False)
    out = g.transform(lambda s: s.ewm(span=span, adjust=False, min_periods=min_periods).mean())
    out.name = None
    return out


def prim_ts_diff(df, sym_col, x: pd.Series, lookback: int) -> pd.Series:
    g = x.groupby(df[sym_col], sort=False)
    out = g.diff(lookback)
    out.name = None
    return out


def prim_diff(a: pd.Series, b: pd.Series) -> pd.Series:
    return a - b


def prim_rdiff(a: pd.Series, b: pd.Series, eps: float = 1e-12) -> pd.Series:
    return (a / b) - 1


def prim_zscore(x: pd.Series, mu: pd.Series, sigma: pd.Series, eps: float = 1e-12) -> pd.Series:
    return (x - mu) / (sigma + eps)


def prim_ratio(a: pd.Series, b: pd.Series, eps: float = 1e-12) -> pd.Series:
    return a / (b + eps) 


def prim_log(x: pd.Series, eps: float = 1e-12) -> pd.Series:
    return np.log(np.maximum(x, eps))


def prim_scale(x: pd.Series, scaler: float) -> pd.Series:
    return x * scaler


def prim_tr(df, sym_col, high: pd.Series, low: pd.Series, close: pd.Series) -> pd.Series:
    prev_close = close.groupby(df[sym_col], sort=False).shift(1)
    hl = (high - low).to_numpy()
    hc = (high - prev_close).abs().to_numpy()
    lc = (low - prev_close).abs().to_numpy()
    out = np.maximum.reduce([hl, hc, lc])
    return pd.Series(out, index=df.index)


def post_clip(x: pd.Series, lo: Optional[float] = None, hi: Optional[float] = None) -> pd.Series:
    return x.clip(lo, hi)


def post_log(x: pd.Series, eps: float = 1e-12) -> pd.Series:
    return np.log(np.maximum(x, eps))


def post_index_scale(x: pd.Series, offset: float = 100.0) -> pd.Series:
    return offset - (offset / (1 + x))


def post_scale(x: pd.Series, scaler: float) -> pd.Series:
    return x * scaler


def post_cs_rank(df: pd.DataFrame, date_col: str, x: pd.Series) -> pd.Series:
    return x.groupby(df[date_col], sort=False).rank(pct=True)


def post_cs_zscore(df: pd.DataFrame, date_col: str, x: pd.Series, eps: float = 1e-12) -> pd.Series:
    g = x.groupby(df[date_col], sort=False)
    mu = g.transform("mean")
    sd = g.transform("std")
    return (x - mu) / (sd + eps)


PRIMITIVES: Dict[str, Callable[..., pd.Series]] = {
    "ts_return": prim_ts_return,
    "ts_mean": prim_ts_mean,
    "ts_std": prim_ts_std,
    "ts_max": prim_ts_max,
    "ts_min": prim_ts_min,
    "ts_ewm_mean": prim_ts_ewm_mean,
    "ts_ewm_span_mean": prim_ts_ewm_span_mean,
    "ts_diff": prim_ts_diff,
    "diff": prim_diff,
    "rdiff": prim_rdiff,
    "scale": prim_scale,
    "zscore": prim_zscore,
    "ratio": prim_ratio,
    "log": prim_log,
    "tr": prim_tr,
}

POSTS: Dict[str, Callable[..., pd.Series]] = {
    "clip": post_clip,
    "log": post_log,
    "index_scale": post_index_scale,
    "scale": post_scale,
    "cs_rank": post_cs_rank,
    "cs_zscore": post_cs_zscore,
}


# ---------- Function call helpers ----------
def prepare_call(fn: Callable, available_inputs: dict) -> dict:
    fn_sig = inspect.signature(fn)
    fn_params = fn_sig.parameters

    call_kwargs = {}
    missing_required = []

    for name, p in fn_params.items():
        if name in available_inputs:
            call_kwargs[name] = available_inputs[name]
        else:
            required = (
                p.default is inspect._empty
                and p.kind in (
                    inspect.Parameter.POSITIONAL_OR_KEYWORD,
                    inspect.Parameter.KEYWORD_ONLY
                )
            )
            if required:
                missing_required.append(name)

    if len(missing_required) > 0:
        return missing_required, 300

    return call_kwargs, 200


# ---------- Builder ----------
class FeatureBuilder:
    def __init__(
        self,
        df: pd.DataFrame,
        *,
        date_col: str = 'date',
        sym_col: str = 'symbol',
        eps: float = 1e-12,
    ):
        self.df = df
        self.date_col = date_col
        self.sym_col = sym_col
        self.eps = eps
        self._cache: Dict[Tuple, pd.Series] = {}

    def _resolve(self, v: Any, computed: Dict[str, pd.Series]) -> Any:
        # Series literal
        if isinstance(v, pd.Series):
            return v

        # reference string: either computed node or df column
        if isinstance(v, str):
            if v in computed:
                return computed[v]
            if v in self.df.columns:
                s = self.df[v].copy()
                s.name = v
                return s

        # literal (int/float/etc)
        return v

    def build(self, specs: Iterable[FeatureSpec]) -> Dict[str, pd.Series]:
        if not isinstance(specs, Iterable):
            specs = [specs]
        specs = list(specs)
        spec_map = {s.name: s for s in specs}

        computed: Dict[str, pd.Series] = {}
        remaining = set(spec_map.keys())

        # iterate-until-done dependency resolution
        progressed = True
        while remaining and progressed:
            progressed = False

            for name in list(remaining):
                spec = spec_map[name]

                # resolve inputs; if any input references an unbuilt spec, defer
                resolved_inputs = {}
                blocked = False
                for k, v in (spec.inputs or {}).items():
                    if isinstance(v, str) and (v in spec_map) and (v not in computed):
                        blocked = True
                        break
                    resolved_inputs[k] = self._resolve(v, computed)

                if blocked:
                    continue

                # resolve params
                params = spec.params or {}

                # prepare available inputs
                # priority: builder context < provided params < provided inputs
                available_inputs = {
                    "df": self.df,
                    "sym_col": self.sym_col,
                    "date_col": self.date_col,
                    "eps": self.eps,
                    **params,
                    **resolved_inputs,
                }

                # grab feature if cached, if not build feature then cache
                cache_key = (spec.primitive, spec.name, tuple(sorted(params.items())))
                if cache_key in self._cache:
                    out = self._cache[cache_key]
                else:
                    prim = PRIMITIVES[spec.primitive]

                    # prepare call_kwargs
                    call_kwargs, status = prepare_call(
                        fn=prim,
                        available_inputs=available_inputs
                    )
                    if status == 300:
                        raise ValueError(
                            f"While building feature '{spec.name}', primitive '{spec.primitive}' "
                            f"is missing required inputs {call_kwargs}. "
                            f"Available keys: {sorted(available_inputs.keys())}"
                        )

                    # pass eps only if primitive accepts it
                    # prim_fn_params = inspect.signature(prim).parameters
                    # if "eps" in prim_fn_params and "eps" not in params:
                    #     params = dict(params)
                    #     params["eps"] = self.eps

                    # execute primitive function
                    out = prim(**call_kwargs)
                    if not isinstance(out, pd.Series):
                        out = pd.Series(out, index=self.df.index)
                    out.name = spec.name
                    self._cache[cache_key] = out

                # post transforms
                if spec.post:
                    for post_name, post_params in spec.post:
                        fn = POSTS[post_name]
                        pp = dict(post_params)

                        post_available_inputs = {
                            "df": self.df,
                            "sym_col": self.sym_col,
                            "date_col": self.date_col,
                            "eps": self.eps,
                            "x": out,
                            **pp,
                        }

                        post_call_kwargs, post_status = prepare_call(
                            fn=fn,
                            available_inputs=post_available_inputs
                        )
                        if post_status == 300:
                            raise ValueError(
                                f"While building feature '{spec.name}', post transform '{post_name}' "
                                f"is missing required inputs {post_call_kwargs}. "
                                f"Available keys: {sorted(post_available_inputs.keys())}"
                            )
                        out = fn(**post_call_kwargs)
                    out.name = spec.name

                computed[name] = out
                remaining.remove(name)
                progressed = True

        if remaining:
            raise ValueError(f"Unresolved specs (cycles or missing deps): {sorted(remaining)}")

        return computed

    def build_published(self, specs: Iterable[FeatureSpec]) -> Dict[str, pd.Series]:
        if not isinstance(specs, Iterable):
            specs = [specs]
        computed = self.build(specs)
        return {s.name: computed[s.name] for s in specs if s.publish}

In [37]:
fn_sig = inspect.signature(prim_ts_mean)
fn_sig

<Signature (df, sym_col, x: 'pd.Series', window: 'int', min_periods: 'Optional[int]' = None) -> 'pd.Series'>

In [38]:
fn_params = fn_sig.parameters
fn_params

mappingproxy({'df': <Parameter "df">,
              'sym_col': <Parameter "sym_col">,
              'x': <Parameter "x: 'pd.Series'">,
              'window': <Parameter "window: 'int'">,
              'min_periods': <Parameter "min_periods: 'Optional[int]' = None">})

In [39]:
for p_name, p in fn_params.items():
    if p.default is inspect._empty and p.kind in (inspect.Parameter.POSITIONAL_OR_KEYWORD, inspect.Parameter.KEYWORD_ONLY):
        print(p_name, ' is required')
    else:
        print(p_name, ' is not required')

df  is required
sym_col  is required
x  is required
window  is required
min_periods  is not required


In [46]:
test_spec = FeatureSpec(
    name="temp_ret_1d",
    primitive="ts_return",
    inputs={"price": "adjClose"},
    params={"lookback": 1},
    publish=True
)

In [47]:
builder = FeatureBuilder(df)

builder.build_published([test_spec])

{'temp_ret_1d': 0              NaN
 1         0.000000
 2         0.158224
 3         0.047628
 4         0.000000
             ...   
 275928   -0.003744
 275929   -0.002253
 275930    0.004875
 275931    0.000000
 275932    0.000000
 Name: temp_ret_1d, Length: 275933, dtype: float64}

# Naming Rules

Domain = the data source or economic origin of the information. (px, liq, fin, risk, etc.)  
Family = the economic or statistical concept. (ret, atr, rsi, beta, etc.)  
Signal = the mathematical form of the feature. (logret, mean, std, z, ratio, norm, diff, etc.)  

In [8]:
nameConvention = {
    'domain': {
        'px': 'price / market data',
        'vol': 'volume / liquidity',
        'fin': 'financial statement / fundamentals',
        'risk': 'risk model or tail-risk derived',
        'mkt': 'market baseline / benchmark related',
        'evt': 'event-driven',
        'alt': 'alternative data',
        'tmp': 'internal temporary node',
    },
    'family': {
        'ret': 'return',
        'mom': '',
        'atr': 'average true range',
        'rsi': 'relative strength index',
        'beta': 'beta',
        'dd': 'drawdown',
        'val': 'value',
        'qlty': 'quality',
        'gwth': 'growth',
        'lvg': 'leverage',
    },
    'signal': {
        'logret',
        'mean',
        'std',
        'z',
        'norm',
        'ratio',
        'diff',
        'norm',
        'rankcs',
        'zcs',
        'cvar',
    },
    'params': {
        'lb20': '20-day lookback',
        'lb1-w20': '',
        'w14': '',
        'w14-ref60': '',
        'p05-w60': '',
        'spy-w126': '',
    },
    'transform': {
        'raw': '',
        'clip5': '',
        'rankcs': '',
        'zcs': '',
        'log': '',
        'winsor1': '',
        'none': '',
    }
}

In [27]:
def family_ts_ret(price_col: str, lookbacks: List[int]) -> List[FeatureSpec]:
    specs = [
        FeatureSpec(
            name=f"px__ret__logret__lb{lb}__raw",
            primitive="ts_return",
            inputs={"price": price_col},
            params={"lookback": lb},
            publish=True,
        ) for lb in lookbacks
    ]
    return specs


def family_ts_mar(
    price_col: str,
    lookbacks: List[int],
    rollwindow: List[int],
    combinator: Combinator,
) -> List[FeatureSpec]:
    specs = []
    for lb, w in combinator(lookbacks, rollwindow):
        ret = f"px__ret__logret__lb{lb}__raw"
        mar = f"px__ret__logret_mean__lb{lb}_w{w}__raw"
        specs += [
            FeatureSpec(
                name=ret,
                primitive="ts_return",
                inputs={"price": price_col},
                params={"lookback": lb},
                publish=True,
            ),
            FeatureSpec(
                name=mar,
                primitive="ts_mean",
                inputs={"x": ret},
                params={"window": w},
                publish=True,
            ),
        ]
    return specs

# Feature Template

In [5]:
from __future__ import annotations
from dataclasses import dataclass, field
from typing import Callable, Any, Optional
import inspect


@dataclass
class FeatureTemplate:
    name: str
    domain: str
    family: str
    signal: Optional[str]
    template_fn: Callable[..., list]
    description: str = ""
    tags: tuple[str, ...] = field(default_factory=tuple)
    unitless: bool = True

    def __call__(self, *args, **kwargs):
        return self.template_fn(*args, **kwargs)

    @property
    def full_name(self) -> str:
        parts = [self.domain, self.family]
        if self.signal:
            parts.append(self.signal)
        return "__".join(parts)

    @property
    def signature(self):
        return inspect.signature(self.template_fn)

    @property
    def param_names(self) -> list[str]:
        return list(self.signature.parameters.keys())



## Return family

In [6]:
def template_ts_return(price_col: str, lookback: int) -> list[FeatureSpec]:
    return [
        FeatureSpec(
            name=f"px__ret__logret__lb{lookback}__raw",
            primitive="ts_return",
            inputs={"price": price_col},
            params={"lookback": lookback},
            publish=True,
        )
    ]


def template_ts_mar(
    price_col: str,
    lookback: int,
    window: int
) -> list[FeatureSpec]:
    ret = f"px__ret__logret__lb{lookback}__raw"
    mar = f"px__ret__mean__lb{lookback}_w{window}__raw"

    return [
        FeatureSpec(
            name=ret,
            primitive="ts_return",
            inputs={"price": price_col},
            params={"lookback": lookback},
            publish=True,
        ),
        FeatureSpec(
            name=mar,
            primitive="ts_mean",
            inputs={"x": ret},
            params={"window": window},
            publish=True,
        )
    ]


def template_ts_mvr(
    price_col: str,
    lookback: int,
    window: int
) -> list[FeatureSpec]:
    ret = f"px__ret__logret__lb{lookback}__raw"
    mvr = f"px__ret__std__lb{lookback}_w{window}__raw"

    return [
        FeatureSpec(
            name=ret,
            primitive="ts_return",
            inputs={"price": price_col},
            params={"lookback": lookback},
            publish=True,
        ),
        FeatureSpec(
            name=mvr,
            primitive="ts_std",
            inputs={"x": ret},
            params={"window": window},
            publish=True,
        )
    ]


def template_ts_mrz(
    price_col: str,
    lookback: int,
    window: int
) -> list[FeatureSpec]:
    ret = f"px__ret__logret__lb{lookback}__raw"
    mu = f"px__ret__mean__lb{lookback}_w{window}__raw"
    sd = f"px__ret__std__lb{lookback}_w{window}__raw"
    mrz = f"px__ret__z__lb{lookback}_w{window}__clip"

    return [
        FeatureSpec(
            name=ret,
            primitive="ts_return",
            inputs={"price": price_col},
            params={"lookback": lookback},
            publish=True,
        ),
        FeatureSpec(
            name=mu,
            primitive="ts_mean",
            inputs={"x": ret},
            params={"window": window},
            publish=True,
        ),
        FeatureSpec(
            name=sd,
            primitive="ts_std",
            inputs={"x": ret},
            params={"window": window},
            publish=True,
        ),
        FeatureSpec(
            name=mrz,
            primitive="zscore",
            inputs={"x": ret, "mu": mu, "sigma": sd},
            params={"eps": 1e-12},
            post={'clip': {'lo': -5.0, 'hi': 5.0}},
            publish=True,
        ),
    ]


def template_ts_mrr(
    price_col: str,
    lookback: int,
    window: int
) -> list[FeatureSpec]:
    ret = f"px__ret__logret__lb{lookback}__raw"
    mu = f"px__ret__mean__lb{lookback}_w{window}__raw"
    mrr = f"px__ret__r__lb{lookback}_w{window}__raw"

    return [
        FeatureSpec(
            name=ret,
            primitive="ts_return",
            inputs={"price": price_col},
            params={"lookback": lookback},
            publish=True,
        ),
        FeatureSpec(
            name=mu,
            primitive="ts_mean",
            inputs={"x": ret},
            params={"window": window},
            publish=True,
        ),
        FeatureSpec(
            name=mrr,
            primitive="ratio",
            inputs={"a": ret, "b": mu},
            params={"eps": 1e-12},
            publish=True,
        ),
    ]


RETURN_FAMILY = {
    'RETURN': FeatureTemplate(
        name="ts_return",
        domain="px",
        family="ret",
        signal="logret",
        template_fn=template_ts_return,
        description="Time-series log return from price over given lookback window.",
        tags=("price", "return", "timeseries"),
    ),
    'MOVING_AVERAGE_RETURN': FeatureTemplate(
        name="ts_mar",
        domain="px",
        family="ret",
        signal="mean",
        template_fn=template_ts_mar,
        description="moving average of log return",
        tags=("price", "return", "average", "timeseries"),
    ),
    'MOVING_VOLATILITY_RETURN': FeatureTemplate(
        name="ts_mvr",
        domain="px",
        family="ret",
        signal="std",
        template_fn=template_ts_mvr,
        description="moving std of log return",
        tags=("price", "return", "std", "volatility", "timeseries"),
    ),
    'RETURN_ZSCORE': FeatureTemplate(
        name="ts_mzr",
        domain="px",
        family="ret",
        signal="z",
        template_fn=template_ts_mrz,
        description="zscore of return in moving window",
        tags=("price", "return", "zscore", "timeseries"),
    ),
    'RETURN_RATIO': FeatureTemplate(
        name="ts_mrr",
        domain="px",
        family="ret",
        signal="r",
        template_fn=template_ts_mrr,
        description="return to its average ratio in moving window",
        tags=("price", "return", "ratio", "timeseries"),
    ),
}

## RawPrice family

In [7]:
def template_ts_map(
    price_col: str,
    window: int
) -> list[FeatureSpec]:
    maprice = f"px__prc__mean__w{window}__raw"

    return [
        FeatureSpec(
            name=maprice,
            primitive="ts_mean",
            inputs={"x": price_col},
            params={"window": window},
            publish=False,
        )
    ]


def template_ts_mvp(
    price_col: str,
    window: int
) -> list[FeatureSpec]:
    mvprice = f"px__prc__std__w{window}__raw"

    return [
        FeatureSpec(
            name=mvprice,
            primitive="ts_std",
            inputs={"x": price_col},
            params={"window": window},
            publish=False,
        )
    ]


def template_ts_blgz(
    price_col: str,
    window: int
) -> list[FeatureSpec]:
    maprice = f"px__prc__mean__w{window}__raw"
    mvprice = f"px__prc__std__w{window}__raw"
    blgz = f"px__prc__z__w{window}__clip"

    return [
        FeatureSpec(
            name=maprice,
            primitive="ts_mean",
            inputs={"x": price_col},
            params={"window": window},
            publish=False,
        ),
        FeatureSpec(
            name=mvprice,
            primitive="ts_std",
            inputs={"x": price_col},
            params={"window": window},
            publish=False,
        ),
        FeatureSpec(
            name=blgz,
            primitive="zscore",
            inputs={"x": price_col, "mu": maprice, "sigma": mvprice},
            params={"eps": 1e-12},
            post={'clip': {'lo': -5.0, 'hi': 5.0}},
            publish=True,
        )
    ]


def template_ts_mapd(
    price_col: str,
    short_w: int,
    long_w: int,
) -> list[FeatureSpec]:
    short_map = f"px__prc__mean__w{short_w}__log"
    long_map = f"px__prc__mean__w{long_w}__log"
    mapd = f"px__prc__diff__w{short_w}_w{long_w}__log"

    return [
        FeatureSpec(
            name=short_map,
            primitive="ts_mean",
            inputs={"x": price_col},
            params={"window": short_w},
            post={"log": {}},
            publish=False,
        ),
        FeatureSpec(
            name=long_map,
            primitive="ts_mean",
            inputs={"x": price_col},
            params={"window": long_w},
            post={"log": {}},
            publish=False,
        ),
        FeatureSpec(
            name=mapd,
            primitive="diff",
            inputs={"a": short_map, "b": long_map},
            publish=True,
        )
    ]


def template_ts_mapdrtp(
    price_col: str,
    short_w: int,
    long_w: int,
) -> list[FeatureSpec]:
    short_map = f"px__prc__mean__w{short_w}__raw"
    long_map = f"px__prc__mean__w{long_w}__raw"
    mapd = f"px__prc__diff__w{short_w}_w{long_w}__raw"
    mapdrtp = f"px__prc__r__w{short_w}_w{long_w}__raw"

    return [
        FeatureSpec(
            name=short_map,
            primitive="ts_mean",
            inputs={"x": price_col},
            params={"window": short_w},
            publish=False,
        ),
        FeatureSpec(
            name=long_map,
            primitive="ts_mean",
            inputs={"x": price_col},
            params={"window": long_w},
            publish=False,
        ),
        FeatureSpec(
            name=mapd,
            primitive="diff",
            inputs={"a": short_map, "b": long_map},
            publish=False,
        ),
        FeatureSpec(
            name=mapdrtp,
            primitive="ratio",
            inputs={"a": mapd, "b": long_map},
            params={"eps": 1e-12},
            publish=True,
        )
    ]


def template_ts_mapdrtatr(
    price_col: str,
    high_col: str,
    low_col: str,
    short_w: int,
    long_w: int
) -> list[FeatureSpec]:
    short_map = f"px__prc__mean__w{short_w}__raw"
    long_map = f"px__prc__mean__w{long_w}__raw"
    mapd = f"px__prc__diff__w{short_w}_w{long_w}__raw"
    tr = "px__tr__diff__none__raw"
    atr = f"px__tr__mean__w{long_w}__raw"
    mapdrtatr = f"px__prc_tr__r__w{short_w}_w{long_w}__raw"

    return [
        FeatureSpec(
            name=short_map,
            primitive="ts_mean",
            inputs={"x": price_col},
            params={"window": short_w},
            publish=False,
        ),
        FeatureSpec(
            name=long_map,
            primitive="ts_mean",
            inputs={"x": price_col},
            params={"window": long_w},
            publish=False,
        ),
        FeatureSpec(
            name=mapd,
            primitive="diff",
            inputs={"a": short_map, "b": long_map},
            publish=False,
        ),
        FeatureSpec(
            name=tr,
            primitive="tr",
            inputs={"high": high_col, "low": low_col, "close": price_col},
            publish=False,
        ),
        FeatureSpec(
            name=atr,
            primitive="ts_mean",
            inputs={"x": tr},
            params={"window": long_w},
            publish=False,
        ),
        FeatureSpec(
            name=mapdrtp,
            primitive="ratio",
            inputs={"a": mapd, "b": atr},
            params={"eps": 1e-12},
            publish=True,
        )
    ]


def template_ts_mapdrtmvr(
    price_col: str,
    short_w: int,
    long_w: int
) -> list[FeatureSpec]:
    short_map = f"px__prc__mean__w{short_w}__log"
    long_map = f"px__prc__mean__w{long_w}__log"
    mapd = f"px__prc__diff__w{short_w}_w{long_w}__log"
    ret = "px__ret__logret__lb1__raw"
    mvr = f"px__ret__std__lb1_w{long_w}__raw"
    mapdrtmvr = f"px__prc_ret__r__w{short_w}_w{long_w}__log"

    return [
        FeatureSpec(
            name=short_map,
            primitive="ts_mean",
            inputs={"x": price_col},
            params={"window": short_w},
            post={"log": {}},
            publish=False,
        ),
        FeatureSpec(
            name=long_map,
            primitive="ts_mean",
            inputs={"x": price_col},
            params={"window": long_w},
            post={"log": {}},
            publish=False,
        ),
        FeatureSpec(
            name=mapd,
            primitive="diff",
            inputs={"a": short_map, "b": long_map},
            publish=False,
        ),
        FeatureSpec(
            name=ret,
            primitive="ts_return",
            inputs={"price": price_col},
            params={"lookback": 1},
            publish=True,
        ),
        FeatureSpec(
            name=mvr,
            primitive="ts_std",
            inputs={"x": ret},
            params={"window": long_w},
            publish=True,
        ),
        FeatureSpec(
            name=mapdrtmvr,
            primitive="ratio",
            inputs={"a": mapd, "b": mvr},
            params={"eps": 1e-12},
            publish=True,
        )
    ]


RAWPRICE_FAMILY = {
    # 'MOVING_AVERAGE_PRICE': FeatureTemplate(
    #     name="ts_map",
    #     domain="px",
    #     family="prc",
    #     signal="mean",
    #     template_fn=template_ts_map,
    #     description="",
    #     tags=("price", "average", "timeseries"),
    #     unitless=False,
    # ),
    # 'MOVING_VOLATILITY_PRICE': FeatureTemplate(
    #     name="ts_mvp",
    #     domain="px",
    #     family="prc",
    #     signal="std",
    #     template_fn=template_ts_mvp,
    #     description="",
    #     tags=("price", "std", "volatility", "timeseries"),
    #     unitless=False,
    # ),
    'BOLLINGER_ZSCORE': FeatureTemplate(
        name="ts_blgz",
        domain="px",
        family="prc",
        signal="z",
        template_fn=template_ts_blgz,
        description="zscore of close price in moving window",
        tags=("price", "zscore", "timeseries"),
    ),
    'MOVING_AVERAGE_PRICE_DIFF': FeatureTemplate(
        name="ts_mapd",
        domain="px",
        family="prc",
        signal="diff",
        template_fn=template_ts_mapd,
        description="log difference between average prices of short and long term moving window",
        tags=("price", "average", "diff", "timeseries"),
    ),
    'MOVING_AVERAGE_PRICE_DIFF_TO_LONG_RATIO': FeatureTemplate(
        name="ts_mapdrtp",
        domain="px",
        family="prc",
        signal="r",
        template_fn=template_ts_mapdrtp,
        description="short-long average price difference to long term average price ratio",
        tags=("price", "average", "diff", "ratio", "timeseries"),
    ),
    'MOVING_AVERAGE_PRICE_DIFF_TO_ATR_RATIO': FeatureTemplate(
        name="ts_mapdrtatr",
        domain="px",
        family="prc_tr",
        signal="r",
        template_fn=template_ts_mapdrtatr,
        description="short-long average price difference to long term average true range ratio",
        tags=("price", "average", "diff", "ratio", "true range", "timeseries"),
    ),
    'MOVING_AVERAGE_PRICE_DIFF_TO_MVR_RATIO': FeatureTemplate(
        name="ts_mapdrtmvr",
        domain="px",
        family="prc_ret",
        signal="r",
        template_fn=template_ts_mapdrtmvr,
        description="short-long average price difference to long term return volatility ratio",
        tags=("price", "average", "diff", "ratio", "volatility", "timeseries"),
    ),
}

## TrueRange family

In [8]:
def template_tr(
    high_col: str,
    low_col: str,
    close_col: str
) -> list[FeatureSpec]:
    tr = "px__tr__diff__none__raw"

    return [
        FeatureSpec(
            name=tr,
            primitive="tr",
            inputs={"high": high_col, "low": low_col, "close": close_col},
            publish=False,
        )
    ]


def template_ts_atr(
    high_col: str,
    low_col: str,
    close_col: str,
    window: int
) -> list[FeatureSpec]:
    tr = "px__tr__diff__none__raw"
    atr = f"px__tr__mean__w{window}__raw"

    return [
        FeatureSpec(
            name=tr,
            primitive="tr",
            inputs={"high": high_col, "low": low_col, "close": close_col},
            publish=False,
        ),
        FeatureSpec(
            name=atr,
            primitive="ts_mean",
            inputs={"x": tr},
            params={"window": window},
            publish=False,
        ),
    ]


def template_ts_vtr(
    high_col: str,
    low_col: str,
    close_col: str,
    window: int
) -> list[FeatureSpec]:
    tr = "px__tr__diff__none__raw"
    vtr = f"px__tr__std__w{window}__raw"

    return [
        FeatureSpec(
            name=tr,
            primitive="tr",
            inputs={"high": high_col, "low": low_col, "close": close_col},
            publish=False,
        ),
        FeatureSpec(
            name=vtr,
            primitive="ts_std",
            inputs={"x": tr},
            params={"window": window},
            publish=False,
        ),
    ]


def template_ts_natr(
    high_col: str,
    low_col: str,
    close_col: str,
    window: int
) -> list[FeatureSpec]:
    tr = "px__tr__diff__none__raw"
    atr = f"px__tr__mean__w{window}__raw"
    maprice = f"px__prc__mean__w{window}__raw"
    natr = f"px__tr_prc__norm__w{window}__raw"

    return [
        FeatureSpec(
            name=tr,
            primitive="tr",
            inputs={"high": high_col, "low": low_col, "close": close_col},
            publish=False,
        ),
        FeatureSpec(
            name=atr,
            primitive="ts_mean",
            inputs={"x": tr},
            params={"window": window},
            publish=False,
        ),
        FeatureSpec(
            name=maprice,
            primitive="ts_mean",
            inputs={"x": close_col},
            params={"window": window},
            publish=False,
        ),
        FeatureSpec(
            name=natr,
            primitive="ratio",
            inputs={"a": atr, "b": maprice},
            params={"eps": 1e-12},
            publish=True,
        ),
    ]


def template_ts_zatr(
    high_col: str,
    low_col: str,
    close_col: str,
    window: int
) -> list[FeatureSpec]:
    tr = "px__tr__diff__none__raw"
    atr = f"px__tr__mean__w{window}__raw"
    mu = f"px__tr__mean2__w{window}__raw"
    sd = f"px__tr__sd__w{window}__raw"
    zatr = f"px__tr__z__w{window}__clip"

    return [
        FeatureSpec(
            name=tr,
            primitive="tr",
            inputs={"high": high_col, "low": low_col, "close": close_col},
            publish=False,
        ),
        FeatureSpec(
            name=atr,
            primitive="ts_mean",
            inputs={"x": tr},
            params={"window": window},
            publish=False,
        ),
        FeatureSpec(
            name=mu,
            primitive="ts_mean",
            inputs={"x": atr},
            params={"window": window},
            publish=True,
        ),
        FeatureSpec(
            name=sd,
            primitive="ts_std",
            inputs={"x": atr},
            params={"window": window, "eps": 1e-12},
            publish=True,
        ),
        FeatureSpec(
            name=zatr,
            primitive="zscore",
            inputs={"x": atr, "mu": mu, "sigma": sd},
            params={"eps": 1e-12},
            post={'clip': {'lo': -5.0, 'hi': 5.0}},
            publish=True,
        )
    ]


def template_ts_ratr(
    high_col: str,
    low_col: str,
    close_col: str,
    window: int
) -> list[FeatureSpec]:
    tr = "px__tr__diff__none__raw"
    atr = f"px__tr__mean__w{window}__raw"
    mu = f"px__tr__mean2__w{window}__raw"
    ratr = f"px__tr__r__w{window}__raw"

    return [
        FeatureSpec(
            name=tr,
            primitive="tr",
            inputs={"high": high_col, "low": low_col, "close": close_col},
            publish=False,
        ),
        FeatureSpec(
            name=atr,
            primitive="ts_mean",
            inputs={"x": tr},
            params={"window": window},
            publish=False,
        ),
        FeatureSpec(
            name=mu,
            primitive="ts_mean",
            inputs={"x": atr},
            params={"window": window},
            publish=False,
        ),
        FeatureSpec(
            name=ratr,
            primitive="ratio",
            inputs={"a": atr, "b": mu},
            params={"eps": 1e-12},
            publish=True,
        ),
    ]


ATR_FAMILY = {
    'NORMALIZED_AVERAGE_TRUERANGE': FeatureTemplate(
        name="ts_natr",
        domain="px",
        family="tr_prc",
        signal="norm",
        template_fn=template_ts_natr,
        description="normalized average true range in moving window",
        tags=("true range", "average", "normalized", "timeseries"),
    ),
    'ZSCORE_AVERAGE_TRUERANGE': FeatureTemplate(
        name="ts_zatr",
        domain="px",
        family="tr",
        signal="z",
        template_fn=template_ts_zatr,
        description="zscore of average true range in moving window",
        tags=("true range", "average", "zscore", "timeseries"),
    ),
    'RATIO_AVERAGE_TRUERANGE': FeatureTemplate(
        name="ts_ratr",
        domain="px",
        family="tr",
        signal="r",
        template_fn=template_ts_ratr,
        description="average true range to its average ratio in moving window",
        tags=("true range", "average", "ratio", "timeseries"),
    ),
}

## RSI family

In [9]:
def template_ts_rsi(
    price_col: str,
    window: int,
) -> list[FeatureSpec]:
    pdiff = "px__prc__tsdiff__lb1__raw"
    gain = "px__rsi__gain__s1__clip"
    loss = "px__rsi__loss__sn1__clip"
    avg_gain = f"px__rsi__gain_mean__w{window}__raw"
    avg_loss = f"px__rsi__loss_mean__w{window}__raw"
    rsi = f"px__rsi__r__w{window}__index"

    return [
        FeatureSpec(
            name=pdiff,
            primitive="ts_diff",
            inputs={"x": price_col},
            params={"lookback": 1},
            publish=False
        ),
        FeatureSpec(
            name=gain,
            primitive="scale",
            inputs={"x": pdiff},
            params={"scaler": 1},
            post={"clip": {"lo": 0}},
            publish=False
        ),
        FeatureSpec(
            name=loss,
            primitive="scale",
            inputs={"x": pdiff},
            params={"scaler": -1},
            post={"clip": {"lo": 0}},
            publish=False
        ),
        FeatureSpec(
            name=avg_gain,
            primitive="ts_ewm_mean",
            inputs={"x": gain},
            params={"window": window},
            publish=False
        ),
        FeatureSpec(
            name=avg_loss,
            primitive="ts_ewm_mean",
            inputs={"x": loss},
            params={"window": window},
            publish=False
        ),
        FeatureSpec(
            name=rsi,
            primitive="ratio",
            inputs={"a": avg_gain, "b": avg_loss},
            params={"eps": 1e-12},
            post={"index_scale": {"offset": 100.0}},
            publish=True,
        ),
    ]


def template_ts_rsiz(
    price_col: str,
    window: int,
    period: int,
) -> list[FeatureSpec]:
    pdiff = "px__prc__tsdiff__lb1__raw"
    gain = "px__rsi__gain__s1__clip"
    loss = "px__rsi__loss__sn1__clip"
    avg_gain = f"px__rsi__gain_mean__w{window}__raw"
    avg_loss = f"px__rsi__loss_mean__w{window}__raw"
    rsi = f"px__rsi__r__w{window}__index"
    mu = f"px__rsi__mean__w{window}_p{period}__index"
    sd = f"px__rsi__std__w{window}_p{period}__index"
    rsiz = f"px__rsi__z__w{window}_p{period}__index"

    return [
        FeatureSpec(
            name=pdiff,
            primitive="ts_diff",
            inputs={"x": price_col},
            params={"lookback": 1},
            publish=False
        ),
        FeatureSpec(
            name=gain,
            primitive="scale",
            inputs={"x": pdiff},
            params={"scaler": 1},
            post={"clip": {"lo": 0}},
            publish=False
        ),
        FeatureSpec(
            name=loss,
            primitive="scale",
            inputs={"x": pdiff},
            params={"scaler": -1},
            post={"clip": {"lo": 0}},
            publish=False
        ),
        FeatureSpec(
            name=avg_gain,
            primitive="ts_ewm_mean",
            inputs={"x": gain},
            params={"window": window},
            publish=False
        ),
        FeatureSpec(
            name=avg_loss,
            primitive="ts_ewm_mean",
            inputs={"x": loss},
            params={"window": window},
            publish=False
        ),
        FeatureSpec(
            name=rsi,
            primitive="ratio",
            inputs={"a": avg_gain, "b": avg_loss},
            params={"eps": 1e-12},
            post={"index_scale": {"offset": 100.0}},
            publish=True,
        ),
        FeatureSpec(
            name=mu,
            primitive="ts_mean",
            inputs={"x": rsi},
            params={"window": period},
            publish=True
        ),
        FeatureSpec(
            name=sd,
            primitive="ts_std",
            inputs={"x": rsi},
            params={"window": period},
            publish=True
        ),
        FeatureSpec(
            name=rsiz,
            primitive="zscore",
            inputs={"x": rsi, "mu": mu, "sigma": sd},
            params={"eps": 1e-12},
            post={'clip': {'lo': -5.0, 'hi': 5.0}},
            publish=True,
        )
    ]


RSI_FAMILY = {
    'RSI': FeatureTemplate(
        name="ts_rsi",
        domain="px",
        family="rsi",
        signal="r",
        template_fn=template_ts_rsi,
        description="relative strength index",
        tags=("price", "rsi", "index", "timeseries"),
    ),
    'RSI_ZSCORE': FeatureTemplate(
        name="ts_rsiz",
        domain="px",
        family="rsi",
        signal="z",
        template_fn=template_ts_rsiz,
        description="zscore of relative strength index",
        tags=("price", "rsi", "index", "zscore", "timeseries"),
    ),
}

## Drawdown family

In [10]:
def template_ts_mdd(
    price_col: str,
    window: int
) -> list[FeatureSpec]:
    peak = f"px__mdd__max__w{window}__raw"
    dd = f"px__mdd__rdiff__w{window}__raw"
    mdd = f"px__mdd__min__w{window}__scale"

    return [
        FeatureSpec(
            name=peak,
            primitive="ts_max",
            inputs={"x": price_col},
            params={"window": window},
            publish=False
        ),
        FeatureSpec(
            name=dd,
            primitive="rdiff",
            inputs={"a": price_col, "b": peak},
            params={"eps": 1e-12},
            publish=False
        ),
        FeatureSpec(
            name=mdd,
            primitive="ts_min",
            inputs={"x": dd},
            params={"window": window},
            post={"scale": {"scaler": -1}},
            publish=True
        ),
    ]


def template_ts_mddatrnorm(
    price_col: str,
    high_col: str,
    low_col: str,
    window: int
) -> list[FeatureSpec]:
    _mdd = template_ts_mdd(price_col, window)
    _atr = template_ts_atr(high_col, low_col, price_col, 14)

    mdd = _mdd[-1].name
    atr = _atr[-1].name
    atr_base = f"px__tr__mean2__w14_p252__raw"
    mddatrnorm = f"px__mdd__atrnorm__w{window}__raw"

    return _mdd + _atr + [
        FeatureSpec(
            name=atr_base,
            primitive="ts_ewm_span_mean",
            inputs={"x": atr},
            params={"span": 252},
            publish=False
        ),
        FeatureSpec(
            name=mddatrnorm,
            primitive="ratio",
            inputs={"a": mdd, "b": atr_base},
            params={"eps": 1e-12},
            publish=True
        ),
    ]


MDD_FAMILY = {
    'MDD': FeatureTemplate(
        name="ts_mdd",
        domain="px",
        family="mdd",
        signal="max",
        template_fn=template_ts_mdd,
        description="maximum drawdown in moving window",
        tags=("price", "mdd", "max", "timeseries"),
    ),
    'MDD_ATR_NORM': FeatureTemplate(
        name="ts_mddatrnorm",
        domain="px",
        family="mdd",
        signal="atrnorm",
        template_fn=template_ts_mddatrnorm,
        description="maximum drawdown in moving window normalized by average true range baseline",
        tags=("price", "mdd", "norm", "atr", "timeseries"),
    ),
}

## Beta family

In [1]:
a = [1, 2, 3]

[*a[:-1], 5]

[1, 2, 5]

In [ ]:

    def create_baseprice(self, symbol):
        colName = f'baseprice_{symbol.lower()}'
        if colName in self.cache['values']:
            return self.cache['values'][colName]

        min_date = self.minDate.strftime("%Y-%m-%d")
        max_date = self.maxDate.strftime("%Y-%m-%d")
        endpoint = 'historical-price-eod/dividend-adjusted'
        data = getFMPData(
            endpoint,
            symbol=symbol.upper(),
            from_=min_date,
            to=max_date
        )
        df = pd.DataFrame(data)
        df['date'] = pd.to_datetime(df['date'])
        df.set_index('date', inplace=True)
        col = df['adjClose'].sort_index()
        self.cache_column(col, colName, 'MarketBaseline')
        return col

    def create_beta(self, w, symbol):
        colName = f'beta_{w}_{symbol.lower()}'
        if colName in self.cache['values']:
            return self.cache['values'][colName]

        if w <= self.max_window:
            mkt_p = self.create_baseprice(symbol)
            asset_df = self.df[['symbol', 'date', 'adjClose']]
            asset_p = pd.pivot(
                asset_df,
                index='date',
                columns='symbol',
                values='adjClose'
            ).sort_index()
            beta_keys = asset_df[['symbol', 'date']].copy()

            mkt_r = np.log1p(mkt_p.pct_change(fill_method=None))
            asset_r = np.log1p(asset_p.pct_change(fill_method=None))
            asset_r_aligned, mkt_r_aligned = asset_r.align(mkt_r, join='inner', axis=0)

            cov = asset_r_aligned.rolling(w, min_periods=w).cov(mkt_r_aligned)
            var = mkt_r_aligned.rolling(w, min_periods=w).var()
            beta_wide = cov.div(var.where(var.abs() > self.eps), axis=0)

            beta_long = beta_wide.stack(future_stack=True).rename(colName).reset_index()
            # beta_long.columns = ['date', 'symbol', colName]
            col = beta_keys.merge(beta_long, on=['symbol', 'date'], how='left', sort=False)[colName]
            self.cache_column(col, colName, self.schema['beta']['name'])
            return col

## Var family

In [ ]:
    def create_var(self, i, j, p):
        colName = f'var_{i}_{j}_p{int(p*100)}'
        if colName in self.cache['values']:
            return self.cache['values'][colName]

        if i + j <= self.max_window and i < j:
            g = self.df[self.rawInput['ticker']]
            ret = self.create_ret(i)
            # col = ret.groupby(g).rolling(j, min_periods=j).quantile(p)
            col = ret.groupby(g).transform(lambda s: s.rolling(j, min_periods=j).quantile(p))
            self.cache_column(col, colName, self.schema['var']['name'])
            return col

    def create_varr(self, i, j, p=0.05):
        colName = f'varr_{i}_{j}_p{int(p*100)}'
        if colName in self.cache['values']:
            return self.cache['values'][colName]

        if i + j <= self.max_window and i < j:
            lower = self.create_var(i, j, p)
            upper = self.create_var(i, j, 1-p)
            col = upper.abs() / (lower.abs() + self.eps)
            self.cache_column(col, colName, self.schema['varr']['name'])
            return col

    def create_cvar(self, i, j, p):
        colName = f'cvar_{i}_{j}_p{int(p*100)}'
        if colName in self.cache['values']:
            return self.cache['values'][colName]

        if i + j <= self.max_window and i < j and j * p >= 2:
            ret = self.create_ret(i)
            ret_long_df = pd.concat([self.df[[self.rawInput['ticker'], 'date']], ret], axis=1)
            ret_wide_df = pd.pivot(ret_long_df, columns=self.rawInput['ticker'], index='date', values=ret.name)
            q_arr = ret_wide_df.rolling(window=j, min_periods=j).quantile(p).to_numpy()
            q_arr = q_arr[(j - 1):, :]
            ret_arr = ret_wide_df.to_numpy()
            ret_arr_roll = sliding_window_view(ret_arr, j, axis=0)
            mask = ret_arr_roll < q_arr[:, :, None]
            sum_ = np.sum(np.where(mask, ret_arr_roll, 0.0), axis=2)
            cnt = np.sum(mask, axis=2)
            out = np.divide(sum_, cnt, out=np.full_like(sum_, np.nan), where=cnt > 0)
            out_wide_df = pd.DataFrame(out, index=ret_wide_df.iloc[(j - 1):].index, columns=ret_wide_df.columns)
            out_wide_df = out_wide_df.reindex(ret_wide_df.index)
            out_long_df = out_wide_df.reset_index().melt(
                id_vars=out_wide_df.index.name,
                var_name=out_wide_df.columns.name,
                value_name=colName
            )
            ret_long_df2 = ret_long_df.merge(out_long_df, how='left', on=[self.rawInput['ticker'], 'date'])
            col = ret_long_df2[colName]
            self.cache_column(col, colName, self.schema['cvar']['name'])
            return col

# Feature Building

In [7]:
df.tail(10)

,symbol,date,adjOpen,adjHigh,adjLow,adjClose,adjVolume,rawClose,rawVolume,exchange,industry,sector,ipoDate,stockPrice,numberOfShares,marketCapitalization,minusCashAndCashEquivalents,addTotalDebt,evEnterpriseValue,price_tr20,dollarVolume,dollarVolume_tr20,turnOver,turnOver_tr20,lowLiquidity,incmFilingDate,incmAcceptedDate,revenue,costOfRevenue,grossProfit,researchAndDevelopmentExpenses,generalAndAdministrativeExpenses,sellingAndMarketingExpenses,sellingGeneralAndAdministrativeExpenses,otherExpenses,operatingExpenses,costAndExpenses,netInterestIncome,interestIncome,interestExpense,incmDepreciationAndAmortization,ebitda,ebit,nonOperatingIncomeExcludingInterest,operatingIncome,totalOtherIncomeExpensesNet,incomeBeforeTax,incomeTaxExpense,netIncomeFromContinuingOperations,netIncomeFromDiscontinuedOperations,otherAdjustmentsToNetIncome,incmNetIncome,netIncomeDeductions,bottomLineNetIncome,eps,epsDiluted,weightedAverageShsOut,weightedAverageShsOutDil,balFilingDate,balAcceptedDate,cashAndCashEquivalents,shortTermInvestments,cashAndShortTermInvestments,netReceivables,balAccountsReceivables,otherReceivables,balInventory,prepaids,otherCurrentAssets,totalCurrentAssets,propertyPlantEquipmentNet,goodwill,intangibleAssets,goodwillAndIntangibleAssets,longTermInvestments,taxAssets,otherNonCurrentAssets,totalNonCurrentAssets,otherAssets,totalAssets,totalPayables,accountPayables,otherPayables,accruedExpenses,shortTermDebt,capitalLeaseObligationsCurrent,taxPayables,deferredRevenue,otherCurrentLiabilities,totalCurrentLiabilities,longTermDebt,capitalLeaseObligationsNonCurrent,deferredRevenueNonCurrent,deferredTaxLiabilitiesNonCurrent,otherNonCurrentLiabilities,totalNonCurrentLiabilities,otherLiabilities,capitalLeaseObligations,totalLiabilities,treasuryStock,preferredStock,commonStock,retainedEarnings,additionalPaidInCapital,accumulatedOtherComprehensiveIncomeLoss,otherTotalStockholdersEquity,totalStockholdersEquity,totalEquity,minorityInterest,totalLiabilitiesAndTotalEquity,totalInvestments,totalDebt,netDebt,cfFilingDate,cfAcceptedDate,cfNetIncome,cfDepreciationAndAmortization,deferredIncomeTax,stockBasedCompensation,changeInWorkingCapital,cfAccountsReceivables,cfInventory,accountsPayables,otherWorkingCapital,otherNonCashItems,netCashProvidedByOperatingActivities,investmentsInPropertyPlantAndEquipment,acquisitionsNet,purchasesOfInvestments,salesMaturitiesOfInvestments,otherInvestingActivities,netCashProvidedByInvestingActivities,netDebtIssuance,longTermNetDebtIssuance,shortTermNetDebtIssuance,netStockIssuance,netCommonStockIssuance,commonStockIssuance,commonStockRepurchased,netPreferredStockIssuance,netDividendsPaid,commonDividendsPaid,preferredDividendsPaid,otherFinancingActivities,netCashProvidedByFinancingActivities,effectOfForexChangesOnCash,netChangeInCash,cashAtEndOfPeriod,cashAtBeginningOfPeriod,operatingCashFlow,capitalExpenditure,freeCashFlow,incomeTaxesPaid,interestPaid,marketCap,kmEnterpriseValue,evToSales,evToOperatingCashFlow,evToFreeCashFlow,evToEBITDA,netDebtToEBITDA,currentRatio,incomeQuality,grahamNetNet,taxBurden,interestBurden,workingCapital,investedCapital,returnOnAssets,operatingReturnOnAssets,returnOnTangibleAssets,returnOnEquity,returnOnInvestedCapital,returnOnCapitalEmployed,earningsYield,freeCashFlowYield,capexToOperatingCashFlow,capexToDepreciation,capexToRevenue,salesGeneralAndAdministrativeToRevenue,researchAndDevelopementToRevenue,stockBasedCompensationToRevenue,intangiblesToTotalAssets,averageReceivables,averagePayables,averageInventory,daysOfSalesOutstanding,daysOfPayablesOutstanding,daysOfInventoryOutstanding,operatingCycle,cashConversionCycle,freeCashFlowToEquity,freeCashFlowToFirm,tangibleAssetValue,netCurrentAssetValue,grahamNumber
275923,ZK,2025-12-10,26.85,26.85,26.72,26.74,115100,26.74,115100.0,NYSE,Auto - Manufacturers,Consumer Cyclical,2024-05-10,216.97,257296840.0,5.582618e+10,7.021000e+09,3.162400e+10,8.042918e+10,26.9460,3077774.00,9.632864e+06,0.000447,0.001394,0,2025-11-17,2025-11-17 06:58:31,3.156200e+10

In [7]:
from util.features.core import FeatureBuilder
from util.features.families.returns import RETURN_FAMILY
from util.features.families.price import RAWPRICE_FAMILY
from util.features.families.truerange import ATR_FAMILY
from util.features.families.beta import BETA_FAMILY
from util.features.families.var import VAR_FAMILY

# df columns:
# symbol, date, adjOpen, adjHigh, adjLow, adjClose, adjVolume, rawClose, rawVolume

df = df.sort_values(["symbol", "date"]).reset_index(drop=True)

builder = FeatureBuilder(
    df,
    date_col="date",
    sym_col="symbol",
)

specs = []

# Return features on adjusted close
specs += RETURN_FAMILY["RETURN"](
    price_col="adjClose",
    lookback=1,
)

specs += RETURN_FAMILY["MOVING_AVERAGE_RETURN"](
    price_col="adjClose",
    lookback=1,
    window=20,
)

specs += RETURN_FAMILY["MOVING_VOLATILITY_RETURN"](
    price_col="adjClose",
    lookback=1,
    window=20,
)

specs += RETURN_FAMILY["RETURN_ZSCORE"](
    price_col="adjClose",
    lookback=1,
    window=20,
)

# Price features on adjusted close
specs += RAWPRICE_FAMILY["BOLLINGER_ZSCORE"](
    price_col="adjClose",
    window=20,
)

specs += RAWPRICE_FAMILY["MOVING_AVERAGE_PRICE_DIFF"](
    price_col="adjClose",
    short_w=20,
    long_w=60,
)

specs += RAWPRICE_FAMILY["MOVING_AVERAGE_PRICE_DIFF_TO_LONG_RATIO"](
    price_col="adjClose",
    short_w=20,
    long_w=60,
)

# True range / ATR features using adjusted OHLC
specs += ATR_FAMILY["NORMALIZED_AVERAGE_TRUERANGE"](
    high_col="adjHigh",
    low_col="adjLow",
    close_col="adjClose",
    window=20,
)

specs += ATR_FAMILY["ZSCORE_AVERAGE_TRUERANGE"](
    high_col="adjHigh",
    low_col="adjLow",
    close_col="adjClose",
    window=20,
)

# Beta - SPY
specs += BETA_FAMILY['MARKET_BETA'](
    price_col="adjClose",
    window=20,
    market_symbol='SPY',
)

# Tail risk
specs += VAR_FAMILY["VALUE_AT_RISK"](
    price_col="adjClose",
    lookback=1,
    window=60,
    p=0.05,
)

specs += VAR_FAMILY["VALUE_AT_RISK_RATIO"](
    price_col="adjClose",
    lookback=1,
    window=60,
    p=0.05,
)

specs += VAR_FAMILY["CONDITIONAL_VALUE_AT_RISK"](
    price_col="adjClose",
    lookback=1,
    window=60,
    p=0.05,
)

features = builder.build_published(specs)

In [8]:
feature_df = pd.DataFrame(features, index=df.index)

final_df = pd.concat(
    [df[["symbol", "date"]], feature_df,],
    axis=1,
)

final_df

,symbol,date,px__ret__logret__lb1__raw,px__ret__mean__lb1_w20__raw,px__ret__std__lb1_w20__raw,px__ret__z__lb1_w20__clip,px__prc__z__w20__clip,px__prc__diff__sw20_lw60__raw,px__prc__r__sw20_lw60__raw,px__tr__map-r__w20__raw,px__tr__z__w20__clip,px__beta__r__w20_mktspy__raw,px__var__q__lb1_w60_p5__raw,px__varr__r__lb1_w60_p5__raw,px__cvar__mean__lb1_w60_p5__raw
0,ABAT,2016-02-24,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ABAT,2016-02-25,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ABAT,2016-02-26,0.158224,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ABAT,2016-02-29,0.047628,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ABAT,2016-03-01,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
275928,ZK,2025-12-17,-0.003744,-0.000615,0.003589,-0.871712,-1.034738,-1.392000,-0.049398,0.005562,-1.321720,0.130319,-0.027254,0.893718,-0.03672
275929,ZK,2025-12-18,-0.002253,-0.001115,0.003013,-0.377650,-1.879195,-1.386000,-0.049248,0.005438,-1.260909,0.094889,-0.027254,0.893718,-0.03672
275930,ZK,2025-12-19,0.004875,-0.000447,0.002763,1.926516,-0.224132,-1.358167,-0.048328,0.005048,-1.288574,0.016952,-0.027254,0.893718,-0.03672
275931,ZK,2025-12-22,0.000000,-0.000224,0.002597,0.086249,-0.144624,-1.329333,-0.047361,0.004450,-1.387995,0.058599,-0.027254,0.893718,-0.03672
